To Do:
- Make List page have subpages (e.g., list of forms, list of forms using a field, see below), make these fileres visible after the intro paragraph
- Make list of forms using a given field with description with embeded question bank info

Thought: make a pdf editor the first step in the weaver. Leverage the name normalizer but allow folks to add and remove fields and rename. 

In [1]:
import pandas as pd
import re
#import PyPDF2
import os
from os import walk
import os.path
from os import path
import shutil

import numpy as np

##import time
from datetime import date

from joblib import dump, load

import urllib.parse

In [274]:
import psutil, time

def governor(wait=0,cpu_trigger=55,mem_trigger=66.66,verbose=0):
    wait_tick = 0.01
    cpu = psutil.cpu_percent()
    mem = psutil.virtual_memory().percent
    if verbose==1:
        print("\n=======================================")
        print(" Usage: CPU {}% & Memory {}%".format(cpu,mem))
        
    if (wait<0):
        if verbose==1:
            print("   Full speed ahead!")
        
    elif (wait>(wait_tick*1000)):    
        #if verbose==1:
        print("    Shut it down!!!")
        #display(Javascript('IPython.notebook.save_checkpoint();'))
        time.sleep(5)
        quit()
    
    elif (cpu > cpu_trigger) | (mem > mem_trigger):
        #if verbose==1:
        print("    Slowing down...")
        time.sleep(wait)
        governor(wait+wait_tick)
    
    if verbose==1:
        print("=======================================\n")

In [275]:
governor(0,55,66.66,1)


 Usage: CPU 10.3% & Memory 68.5%
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...
    Slowing down...


KeyboardInterrupt: 

In [4]:
%%time
import formfyxer as lit

CPU times: user 4.18 s, sys: 1.62 s, total: 5.8 s
Wall time: 46.7 s


In [276]:
def removeSpecial(text):
    text = re.sub('[^a-zA-Z0-9]',"_",text)
    return re.sub('_+',"_",text)

In [277]:
def clean_n(n):
    if pd.notnull(n):
        return int(n)
    else:
        return 0

In [278]:
today = date.today().strftime("%Y-%m-%d")

## Old data

In [279]:
%%time
files_df_old = pd.read_csv("../data/processed/form_data.csv")
print(len(files_df_old))
display(files_df_old[:2])
files_df_old.columns

10281


,id,jurisdiction,source,title,group,url,filename,downloaded,pages,fields,fields_conf,fields_old,f_per_p,reading,list,text,downloaded_on
0,04b3a0734774c02edf8eb9056d23954aa38e96c77c3392...,UT,www.utcourts.gov,Community Service Worksheet Third District Juv...,3rd District Juvenile Court: Forms and Pamphlets,https://www.utcourts.gov/courts/juv/juvsites/3...,COMMUNITY%20SERVICE%20WORKSHEET-FRONT%20AND%20...,2021-11-11,2.0,"['name__1', 'name__2', '*docket_number', '*use...","[0.63, 0.61, 1, 1, 1, 0.61, 0.52, 0.53, 0.6, 0...","['name__1', 'name__2', 'docket_number', 'users...",7.5,10.5,[],COMMUNITY SERVICE WORKSHEET\n THIRD DISTRICT J...,NaN
1,6e420f1b3575cfd8ef94b71977da9e38252e3395a78439...,UT,www.utcourts.gov,Third District Juvenile Court Work Program Ref...,3rd District Juvenile Court: Forms and Pamphlets,https://www.utcourts.gov/courts/juv/juvsites/3...,Work_Crew_Application-2007.pdf,2021-11-11,2.0,"['*docket_number', '*users1_birthdate', 'male'...","[1, 1, 0.69, 0.61, 1, 1, 1, 0.56, 0.68, 0.67, ...","['docket_number', 'users1_birthdate', 'male', ...",19.0,10.5,['GO-00-00-00-00'],THIRD DISTRICT JUVENILE COURT WORK PROGRAM REF...,NaN


CPU times: user 447 ms, sys: 310 ms, total: 757 ms
Wall time: 4.08 s


Index(['id', 'jurisdiction', 'source', 'title', 'group', 'url', 'filename',
       'downloaded', 'pages', 'fields', 'fields_conf', 'fields_old', 'f_per_p',
       'reading', 'list', 'text', 'downloaded_on'],
      dtype='object')

In [280]:
%%time
form_sim_data_old = pd.read_csv("../data/processed/form_sim_data.csv")
print(len(form_sim_data_old))
display(form_sim_data_old[:2])
form_sim_data_old.columns

3162622


,id_1,id_2,title_2,similarity
0,e12318ce9037e92ee8e48963f550e1a3229a17a2437b9e...,eb4e35c3b9f355d63ad14ae71866d44b14e61196763553...,Answer to Complaint to Recover Possession of P...,0.979167
1,e12318ce9037e92ee8e48963f550e1a3229a17a2437b9e...,74723a905136a3e30a9eec9aa9191f5d3e4b800ae319df...,"Answer, Damage/Health Hazard to Property, Land...",0.937500


CPU times: user 2.8 s, sys: 430 ms, total: 3.23 s
Wall time: 10 s


Index(['id_1', 'id_2', 'title_2', 'similarity'], dtype='object')

## New Dats

In [281]:
csv_path = "../data/state_forms/"

In [282]:
%%time
forms_df = pd.read_csv(csv_path+"form_data.csv", index_col=0, encoding="utf-8")
forms_df["id"] = forms_df.index

print(len(forms_df))
display(forms_df[:2])
forms_df.columns

30871


,jurisdiction,source,group,title,url,filename,downloaded,id
ce7ec7991f0283ea319598ff05183e50,MA,www.mass.gov,NaN,\n,https://www.mass.gov/doc/order-to-render-mpc-7...,/download,1,ce7ec7991f0283ea319598ff05183e50
13c091b90b3ee286afef69e55591d6f9,MA,www.mass.gov,NaN,\n,https://www.mass.gov/doc/trust-account-mpc-859...,/download,1,13c091b90b3ee286afef69e55591d6f9


CPU times: user 85.4 ms, sys: 13.6 ms, total: 99 ms
Wall time: 168 ms


Index(['jurisdiction', 'source', 'group', 'title', 'url', 'filename',
       'downloaded', 'id'],
      dtype='object')

In [283]:
%%time
forms_df_sup_1 = pd.read_csv(csv_path+"form_scratch_x_1_spot.csv", index_col=0, encoding="utf-8")

CPU times: user 1.01 s, sys: 302 ms, total: 1.31 s
Wall time: 12.4 s


In [284]:
%%time
forms_df_sup_2 = pd.read_csv(csv_path+"form_scratch_x_2_spot.csv", index_col=0, encoding="utf-8")

CPU times: user 1.01 s, sys: 254 ms, total: 1.26 s
Wall time: 9.32 s


In [285]:
%%time
forms_df_sup_3 = pd.read_csv(csv_path+"form_scratch_x_3_spot.csv", index_col=0, encoding="utf-8")

CPU times: user 1.06 s, sys: 276 ms, total: 1.34 s
Wall time: 10.8 s


In [286]:
forms_df_sup = pd.concat([forms_df_sup_1,forms_df_sup_2,forms_df_sup_3])
#forms_df_sup = forms_df_sup_1.copy()

forms_df_sup.index = forms_df_sup.index.str.replace(r'\.pdf$', '', regex=True)
forms_df_sup = forms_df_sup.rename(columns={"title":"meta title"})
forms_df_sup["id"] = forms_df_sup.index
forms_df_sup = forms_df_sup.reset_index()
forms_df_sup = forms_df_sup.drop_duplicates(subset=["id"])

print(len(forms_df_sup))
display(forms_df_sup[:2])
forms_df_sup.columns

24062


,index,meta title,suggested title,description,category,pages,reading grade level,time to answer,list,avg fields per page,...,passive voice percent,citations per field,citation count,all caps percent,difficult words,difficult word count,difficult word percent,calculation required,full_text,id
0,9fffe321a9fbafdb7964293e8c88e265,Writ of Possession,NaN,NaN,NaN,2.0,13,"[5.3, 1.9]",['GO-00-00-00-00'],5.5,...,0.222222,0.090909,1.0,0.021858,"['numbers', 'reprographics', 'applicable', 'at...",59.0,0.161202,False,Writ of Possession \n\n \n\nin the District ...,9fffe321a9fbafdb7964293e8c88e265
1,9ffd35f8778e21212dd0f319ebb497d2,Instructions For Florida Supreme Court Approve...,NaN,NaN,NaN,9.0,20,"[83.0, 9.7]","['ES-00-00-00-00', 'FA-00-00-00-00']",7.888889,...,0.217391,0.000000,0.0,0.038889,"['related', 'special', 'dissolution', 'initial...",243.0,0.071053,False,INSTRUCTIONS FOR FLORIDA SUPREME COURT APPROVE...,9ffd35f8778e21212dd0f319ebb497d2


Index(['index', 'meta title', 'suggested title', 'description', 'category',
       'pages', 'reading grade level', 'time to answer', 'list',
       'avg fields per page', 'fields', 'fields_conf', 'fields_old',
       'number of sentences', 'sentences per page',
       'number of passive voice sentences', 'passive sentences',
       'number of all caps words', 'citations', 'total fields',
       'slotin percent', 'gathered percent', 'created percent',
       'third party percent', 'passive voice percent', 'citations per field',
       'citation count', 'all caps percent', 'difficult words',
       'difficult word count', 'difficult word percent',
       'calculation required', 'full_text', 'id'],
      dtype='object')

In [287]:
forms_df_merged = forms_df.merge(forms_df_sup, on="id", how="right")
print(len(forms_df_merged))
display(forms_df_merged[:2])
forms_df_merged.columns

24062


,jurisdiction,source,group,title,url,filename,downloaded,id,index,meta title,...,third party percent,passive voice percent,citations per field,citation count,all caps percent,difficult words,difficult word count,difficult word percent,calculation required,full_text
0,HI,www.courts.state.hi.us,NaN,Writ of Possession,https://www.courts.state.hi.us/docs/form/kauai...,5DC54.pdf,1.0,9fffe321a9fbafdb7964293e8c88e265,9fffe321a9fbafdb7964293e8c88e265,Writ of Possession,...,0.0,0.222222,0.090909,1.0,0.021858,"['numbers', 'reprographics', 'applicable', 'at...",59.0,0.161202,False,Writ of Possession \n\n \n\nin the District ...
1,FL,www.flcourts.gov,NaN,\n,https://www.flcourts.gov/content/download/6859...,970%28b%29.pdf,1.0,9ffd35f8778e21212dd0f319ebb497d2,9ffd35f8778e21212dd0f319ebb497d2,Instructions For Florida Supreme Court Approve...,...,0.0,0.217391,0.000000,0.0,0.038889,"['related', 'special', 'dissolution', 'initial...",243.0,0.071053,False,INSTRUCTIONS FOR FLORIDA SUPREME COURT APPROVE...


Index(['jurisdiction', 'source', 'group', 'title', 'url', 'filename',
       'downloaded', 'id', 'index', 'meta title', 'suggested title',
       'description', 'category', 'pages', 'reading grade level',
       'time to answer', 'list', 'avg fields per page', 'fields',
       'fields_conf', 'fields_old', 'number of sentences',
       'sentences per page', 'number of passive voice sentences',
       'passive sentences', 'number of all caps words', 'citations',
       'total fields', 'slotin percent', 'gathered percent', 'created percent',
       'third party percent', 'passive voice percent', 'citations per field',
       'citation count', 'all caps percent', 'difficult words',
       'difficult word count', 'difficult word percent',
       'calculation required', 'full_text'],
      dtype='object')

In [288]:
def count_characters(text):
    return len(str(text))

In [289]:
forms_df_merged["downloaded"]= "2023-03"

In [290]:
import hashlib

def hashme(w):
    h = hashlib.md5(w.encode('utf-8'))
    return h.hexdigest()

fed_froms = pd.read_csv("../data/fed_forms/form_scratch_x_1_spot.csv", index_col=0, encoding="utf-8")
fed_froms = fed_froms.reset_index()
#fed_froms = fed_froms.rename(columns={"index":"id"})
fed_froms["id"] = fed_froms["index"]
fed_froms.loc[fed_froms["title"]=="(Untitled)","title"] = fed_froms["id"]
fed_froms["title"] = fed_froms["title"].replace(r'\.pdf$', '', regex=True)
fed_froms["meta title"] = fed_froms["title"]
#fed_froms = fed_froms.rename(columns={"title":"meta title"})

fed_froms["jurisdiction"] = "ca1"
fed_froms["source"] = "ca1.uscourts.gov"
fed_froms["group"] = np.nan
fed_froms["url"] = "javascript:alert('not available');"
fed_froms["filename"] = fed_froms["id"]
fed_froms["downloaded"] = "2023-04"
fed_froms["id"] = fed_froms["id"].apply(hashme)

fed_froms[['jurisdiction', 'source', 'group', 'title', 'url', 'filename',
       'downloaded', 'id', 'index', 'meta title', 'suggested title',
       'description', 'category', 'pages', 'reading grade level',
       'time to answer', 'list', 'avg fields per page', 'fields',
       'fields_conf', 'fields_old', 'number of sentences',
       'sentences per page', 'number of passive voice sentences',
       'passive sentences', 'number of all caps words', 'citations',
       'total fields', 'slotin percent', 'gathered percent', 'created percent',
       'third party percent', 'passive voice percent', 'citations per field',
       'citation count', 'all caps percent', 'difficult words',
       'difficult word count', 'difficult word percent',
       'calculation required', 'full_text']]

,jurisdiction,source,group,title,url,filename,downloaded,id,index,meta title,...,third party percent,passive voice percent,citations per field,citation count,all caps percent,difficult words,difficult word count,difficult word percent,calculation required,full_text
0,ca1,ca1.uscourts.gov,NaN,Bill of Costs,javascript:alert('not available');,ao133_Bill_of_Cost.pdf,2023-04,4cd34891da97f2dc9271a3559d74e2a7,ao133_Bill_of_Cost.pdf,Bill of Costs,...,0,0.333333,0.0000,0,0.033975,"['entered', 'interpreters', 'itemize', 'servic...",97,0.109853,True,",\n\n \n\nUNITED STATES DISTRICT COURT\nfor th..."
1,ca1,ca1.uscourts.gov,NaN,U.S. Marshals Service Form 285 - Service of Pr...,javascript:alert('not available');,USM 285 Process Receipt and Return.pdf,2023-04,9e839dde6965d468952dd00bdda7ba32,USM 285 Process Receipt and Return.pdf,U.S. Marshals Service Form 285 - Service of Pr...,...,0,0.229167,0.0000,0,0.149144,"['readability', 'expediting', 'executed', 'aut...",100,0.122249,True,USM-285 is a 5-part form. Fill out the form a...
2,ca1,ca1.uscourts.gov,NaN,U.S. Marshals Service Form 285 - Service of Pr...,javascript:alert('not available');,US Marshal Form 285 Process Receipt and Return...,2023-04,fa74e5c8651cc1c537c653f288c00b36,US Marshal Form 285 Process Receipt and Return...,U.S. Marshals Service Form 285 - Service of Pr...,...,0,0.229167,0.0000,0,0.149144,"['readability', 'expediting', 'executed', 'aut...",100,0.122249,True,USM-285 is a 5-part form. Fill out the form a...
3,ca1,ca1.uscourts.gov,NaN,Step by Step Guide to Filing a Civil Action Pr...,javascript:alert('not available');,Step by Step Guide to Filing a Civil Action Pr...,2023-04,b9bdd37aeb287a1cbd8372d3c365234e,Step by Step Guide to Filing a Civil Action Pr...,Step by Step Guide to Filing a Civil Action Pr...,...,0,0.349206,0.3000,12,0.059840,"['relevant', 'meaning', 'injured', 'appointed'...",279,0.092753,False,"4/3/23, 3:14 PM\n\nStep by Step Guide to Filin..."
4,ca1,ca1.uscourts.gov,NaN,SS_Complaint,javascript:alert('not available');,SS_Complaint.pdf,2023-04,8e4aa59b8ba99531160331cd1ae43709,SS_Complaint.pdf,SS_Complaint,...,0,0.083333,0.0625,1,0.029703,"['remedies', 'signature', 'adverse', 'affects'...",44,0.145215,False,COMPLAINT \n \n\n \n\nThe above-named Plaint...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,ca1,ca1.uscourts.gov,NaN,2254 Instructions to Inmates (English Spanish),javascript:alert('not available');,2254 Instructions to Inmates (English Spanish)...,2023-04,8d658185fad20e95719f76b30db16e1f,2254 Instructions to Inmates (English Spanish)...,2254 Instructions to Inmates (English Spanish),...,0,0.000000,0.0000,0,0.000000,[],0,0.000000,False,
97,ca1,ca1.uscourts.gov,NaN,1983 Notice to Inmates (English Spanish),javascript:alert('not available');,1983 Notice to Inmates (English Spanish).pdf,2023-04,2d6eebf019d539ab3c24009724e7c079,1983 Notice to Inmates (English Spanish).pdf,1983 Notice to Inmates (English Spanish),...,0,0.000000,0.0000,0,0.000000,[],0,0.000000,False,
98,ca1,ca1.uscourts.gov,NaN,1983 Instructions to Inmates (English Spanish),javascript:alert('not available');,1983 Instructions to Inmates (English Spanish)...,2023-04,0090296bf59dd7937947ce57a8f1f0e3,1983 Instructions to Inmates (English Spanish)...,1983 Instructions to Inmates (English Spanish),...,0,0.000000,0.0000,0,0.000000,[],0,0.000000,False,
99,ca1,ca1.uscourts.gov,NaN,1331 Instructions to Inmates (English),javascript:alert('not available');,1331 Instructions to Inmates (English).pdf,2023-04,4181fb4aaedd78a911d19e518293af6c,1331 Instructions to Inmates (English).pdf,1331 Instructions to Inmates (English),...,0,0.000000,0.0000,0,0.000000,[],0,0.000000,False,


In [291]:
csv_df = forms_df_merged.copy()

In [292]:
csv_df = pd.concat([csv_df,fed_froms])

In [293]:
files_df_old["title_old"] = files_df_old["title"]
len(files_df_old)

10281

In [294]:
csv_df = csv_df.merge(files_df_old[["id","title_old"]], how="left")#.reset_index()
csv_df["title_old"] = csv_df["title_old"].fillna("")


In [295]:
#csv_df.drop_duplicates(inplace=True)

In [296]:
csv_df["title_old_count"] = None
csv_df["title_count"] = None
csv_df["meta_title_count"] = None
csv_df["title_old_count"] = csv_df["title_old"].apply(count_characters)
csv_df["title_count"] = csv_df["title"].apply(count_characters)
csv_df["meta_title_count"] = csv_df["meta title"].apply(count_characters)

In [297]:
csv_df.loc[(csv_df["title_count"]<10),"title"] = csv_df["meta title"]
csv_df.loc[pd.isnull(csv_df["title_count"]),"title"] = csv_df["meta title"]
csv_df.loc[csv_df["title_count"]=="","title"] = csv_df["meta title"]
csv_df.loc[(csv_df["meta_title_count"]>20) & (csv_df["meta title"]!="(Untitled)"),"title"] = csv_df["meta title"]
csv_df.loc[(csv_df["title_old"]!=""),"title"] = csv_df["title_old"]
csv_df["title"] = csv_df["title"].fillna("(Untitled)")

In [298]:
csv_df["new_url"] = "https://courtformsonline.org/forms/"+csv_df["id"]+".pdf"
csv_df = csv_df.rename(columns={"list":"list_codes","reading grade level":"reading_level",
                                "fields": "standardized_fields","url":"original_url",
                                "fields_old":"original_fields","avg fields per page":"f_per_p",
                                "full_text":"text",
                                "downloaded":"downloaded_on"})

In [299]:
csv_df[["id","title","meta title","group","pages","reading_level","downloaded_on","original_url","original_fields","new_url","standardized_fields","jurisdiction"]]


,id,title,meta title,group,pages,reading_level,downloaded_on,original_url,original_fields,new_url,standardized_fields,jurisdiction
0,9fffe321a9fbafdb7964293e8c88e265,Writ of Possession,Writ of Possession,NaN,2.0,13,2023-03,https://www.courts.state.hi.us/docs/form/kauai...,['clerk__district_court_of_the_above_circuit__...,https://courtformsonline.org/forms/9fffe321a9f...,"['clerk_district_court_state', 'duly_authorize...",HI
1,9ffd35f8778e21212dd0f319ebb497d2,Instructions For Florida Supreme Court Approve...,Instructions For Florida Supreme Court Approve...,NaN,9.0,20,2023-03,https://www.flcourts.gov/content/download/6859...,"['JUDICIAL CIRCUIT', 'COUNTY FLORIDA', 'Case n...",https://courtformsonline.org/forms/9ffd35f8778...,"['judicia_l_circuit', 'count_florida', '*docke...",FL
2,9ffba05824a302bc69c7fded6244af49,GAL - Fee Waiver Application (Family),GAL - Fee Waiver Application (Family),NaN,3.0,10,2023-03,https://www.mncourts.gov/mncourtsgov/media/Cou...,"['petitioner', 'respondent', 'if_a_dependent_l...",https://courtformsonline.org/forms/9ffba05824a...,"['*petitioners1_name', '*respondents1_name', '...",MN
3,9ff5c8ab7729bc4343ab717c29dba495,Financial Institution Receipt of Letters,STATE OF NEBRASKA,NaN,2.0,8,2023-03,https://supremecourt.nebraska.gov/sites/defaul...,"['fullcountystatement222', 'typeofcourt222', '...",https://courtformsonline.org/forms/9ff5c8ab772...,"['fullcountystatement', 'typeofcourt', 'county...",NE
4,9ff5c8ab7729bc4343ab717c29dba495,Financial Institution Receipt of Letters,STATE OF NEBRASKA,NaN,2.0,8,2023-03,https://supremecourt.nebraska.gov/sites/defaul...,"['fullcountystatement222', 'typeofcourt222', '...",https://courtformsonline.org/forms/9ff5c8ab772...,"['fullcountystatement', 'typeofcourt', 'county...",NE
...,...,...,...,...,...,...,...,...,...,...,...,...
24815,8d658185fad20e95719f76b30db16e1f,2254 Instructions to Inmates (English Spanish),2254 Instructions to Inmates (English Spanish),NaN,2.0,0,2023-04,javascript:alert('not available');,"['page_0_check_0', 'page_0_check_1', 'page_0_c...",https://courtformsonline.org/forms/8d658185fad...,"['page_check__1', 'page_check__2', 'page_check...",ca1
24816,2d6eebf019d539ab3c24009724e7c079,1983 Notice to Inmates (English Spanish),1983 Notice to Inmates (English Spanish),NaN,4.0,0,2023-04,javascript:alert('not available');,"['page_0_check_0', 'page_0_check_1', 'page_0_c...",https://courtformsonline.org/forms/2d6eebf019d...,"['page_check__1', 'page_check__2', 'page_check...",ca1
24817,0090296bf59dd7937947ce57a8f1f0e3,1983 Instructions to Inmates (English Spanish),1983 Instructions to Inmates (English Spanish),NaN,2.0,0,2023-04,javascript:alert('not available');,"['page_0_check_0', 'page_0_check_1', 'page_0_c...",https://courtformsonline.org/forms/0090296bf59...,"['page_check__1', 'page_check__2', 'page_check...",ca1
24818,4181fb4aaedd78a911d19e518293af6c,1331 Instructions to Inmates (English),1331 Instructions to Inmates (English),NaN,3.0,0,2023-04,javascript:alert('not available');,[],https://courtformsonline.org/forms/4181fb4aaed...,[],ca1


In [300]:
csv_df.columns

Index(['jurisdiction', 'source', 'group', 'title', 'original_url', 'filename',
       'downloaded_on', 'id', 'index', 'meta title', 'suggested title',
       'description', 'category', 'pages', 'reading_level', 'time to answer',
       'list_codes', 'f_per_p', 'standardized_fields', 'fields_conf',
       'original_fields', 'number of sentences', 'sentences per page',
       'number of passive voice sentences', 'passive sentences',
       'number of all caps words', 'citations', 'total fields',
       'slotin percent', 'gathered percent', 'created percent',
       'third party percent', 'passive voice percent', 'citations per field',
       'citation count', 'all caps percent', 'difficult words',
       'difficult word count', 'difficult word percent',
       'calculation required', 'text', 'title_old', 'title_old_count',
       'title_count', 'meta_title_count', 'new_url'],
      dtype='object')

In [301]:
files_df = csv_df.copy()
files_df = files_df.rename(columns={"avg fields per page":"f_per_p","reading grade level":"reading",
                                "full_text":"text",'downloaded_on':'downloaded',
                                   'original_url':'url', 'standardized_fields':'fields', 'original_fields':'fields_old', 'reading_level':'reading', 
                                    'list_codes':'list'
                                   })

In [302]:
files_df["downloaded_on"] = files_df["downloaded"]

In [303]:
files_df = files_df[['id', 'jurisdiction', 'source', 'title', 'group', 'url', 'filename',
       'downloaded', 'pages', 'fields', 'fields_conf', 'fields_old', 'f_per_p',
       'reading', 'list', 'text', 'downloaded_on']]

In [309]:
files_df

,id,jurisdiction,source,title,group,url,filename,downloaded,pages,fields,fields_conf,fields_old,f_per_p,reading,list,text,downloaded_on
0,9fffe321a9fbafdb7964293e8c88e265,HI,www.courts.state.hi.us,Writ of Possession,NaN,https://www.courts.state.hi.us/docs/form/kauai...,5DC54.pdf,2023-03,2.0,"['clerk_district_court_state', 'duly_authorize...","[0.54, 0.46, 0.35000000000000003, 0.36, 0.36, ...",['clerk__district_court_of_the_above_circuit__...,5.5,13,['GO-00-00-00-00'],Writ of Possession \n\n \n\nin the District ...,2023-03
1,9ffd35f8778e21212dd0f319ebb497d2,FL,www.flcourts.gov,Instructions For Florida Supreme Court Approve...,NaN,https://www.flcourts.gov/content/download/6859...,970%28b%29.pdf,2023-03,9.0,"['judicia_l_circuit', 'count_florida', '*docke...","[0.59, 0.42, 1.0, 0.38, 0.34, 1.0, 1.0, 0.4700...","['JUDICIAL CIRCUIT', 'COUNTY FLORIDA', 'Case n...",7.888889,20,"['ES-00-00-00-00', 'FA-00-00-00-00']",INSTRUCTIONS FOR FLORIDA SUPREME COURT APPROVE...,2023-03
2,9ffba05824a302bc69c7fded6244af49,MN,www.mncourts.gov,GAL - Fee Waiver Application (Family),NaN,https://www.mncourts.gov/mncourtsgov/media/Cou...,IFP502F.pdf?ext=.pdf,2023-03,3.0,"['*petitioners1_name', '*respondents1_name', '...","[1.0, 1.0, 0.49, 0.5, 0.49, 0.42, 0.38, 0.39, ...","['petitioner', 'respondent', 'if_a_dependent_l...",12.666667,10,['GO-00-00-00-00'],CONFIDENTIAL \n\nState of Minnesota \nCounty \...,2023-03
3,9ff5c8ab7729bc4343ab717c29dba495,NE,supremecourt.nebraska.gov,Financial Institution Receipt of Letters,NaN,https://supremecourt.nebraska.gov/sites/defaul...,CC-16-2-6-1.pdf,2023-03,2.0,"['fullcountystatement', 'typeofcourt', 'county...","[0.97, 0.35000000000000003, 0.3500000000000000...","['fullcountystatement222', 'typeofcourt222', '...",2.5,8,"['ES-02-00-00-00', 'ES-00-00-00-00']",Nebraska State Court Form \nREQUIRED \nCC 16:2...,2023-03
4,9ff5c8ab7729bc4343ab717c29dba495,NE,supremecourt.nebraska.gov,Financial Institution Receipt of Letters,NaN,https://supremecourt.nebraska.gov/sites/defaul...,CC-16-2-6-1.pdf,2023-03,2.0,"['fullcountystatement', 'typeofcourt', 'county...","[0.97, 0.35000000000000003, 0.3500000000000000...","['fullcountystatement222', 'typeofcourt222', '...",2.5,8,"['ES-02-00-00-00', 'ES-00-00-00-00']",Nebraska State Court Form \nREQUIRED \nCC 16:2...,2023-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24815,8d658185fad20e95719f76b30db16e1f,ca1,ca1.uscourts.gov,2254 Instructions to Inmates (English Spanish),NaN,javascript:alert('not available');,2254 Instructions to Inmates (English Spanish)...,2023-04,2.0,"['page_check__1', 'page_check__2', 'page_check...","[0.51, 0.33, 0.33, 0.33, 0.33, 0.33, 0.33, 0.2...","['page_0_check_0', 'page_0_check_1', 'page_0_c...",5.5,0,[],,2023-04
24816,2d6eebf019d539ab3c24009724e7c079,ca1,ca1.uscourts.gov,1983 Notice to Inmates (English Spanish),NaN,javascript:alert('not available');,1983 Notice to Inmates (English Spanish).pdf,2023-04,4.0,"['page_check__1', 'page_check__2', 'page_check...","[0.51, 0.33, 0.33, 0.33, 0.26]","['page_0_check_0', 'page_0_check_1', 'page_0_c...",1.25,0,[],,2023-04
24817,0090296bf59dd7937947ce57a8f1f0e3,ca1,ca1.uscourts.gov,1983 Instructions to Inmates (English Spanish),NaN,javascript:alert('not available');,1983 Instructions to Inmates (English Spanish)...,2023-04,2.0,"['page_check__1', 'page_check__2', 'page_check...","[0.51, 0.33, 0.33, 0.33, 0.33, 0.33, 0.26, 0.2...","['page_0_check_0', 'page_0_check_1', 'page_0_c...",6.5,0,[],,2023-04
24818,4181fb4aaedd78a911d19e518293af6c,ca1,ca1.uscourts.gov,1331 Instructions to Inmates (English),NaN,javascript:alert('not available');,1331 Instructions to Inmates (English).pdf,2023-04,3.0,[],[],[],0.0,0,[],,2023-04


In [307]:
test_vec_0 = test_vec
test_vec_0

,id,title,group,list_codes,pages,f_per_p,reading_level,downloaded_on,original_url,original_fields,new_url,standardized_fields,jurisdiction,text,vec,PCA
0,9fffe321a9fbafdb7964293e8c88e265,Writ of Possession,NaN,['GO-00-00-00-00'],2.0,5.5,13,2023-03,https://www.courts.state.hi.us/docs/form/kauai...,['clerk__district_court_of_the_above_circuit__...,https://courtformsonline.org/forms/9fffe321a9f...,"['clerk_district_court_state', 'duly_authorize...",HI,Writ of Possession \n\n \n\nin the District ...,"[-0.027059428771668746, -0.01102143584139791, ...","[0.7096992402661475, -0.1001827335832918, 0.00..."
1,9ffd35f8778e21212dd0f319ebb497d2,Instructions For Florida Supreme Court Approve...,NaN,"['ES-00-00-00-00', 'FA-00-00-00-00']",9.0,7.888888888888889,20,2023-03,https://www.flcourts.gov/content/download/6859...,"['JUDICIAL CIRCUIT', 'COUNTY FLORIDA', 'Case n...",https://courtformsonline.org/forms/9ffd35f8778...,"['judicia_l_circuit', 'count_florida', '*docke...",FL,INSTRUCTIONS FOR FLORIDA SUPREME COURT APPROVE...,"[-0.08607775506534653, -0.020282120776711276, ...","[-0.4827016710945098, -0.16228668440049665, 0...."
2,9ffba05824a302bc69c7fded6244af49,GAL - Fee Waiver Application (Family),NaN,['GO-00-00-00-00'],3.0,12.666666666666666,10,2023-03,https://www.mncourts.gov/mncourtsgov/media/Cou...,"['petitioner', 'respondent', 'if_a_dependent_l...",https://courtformsonline.org/forms/9ffba05824a...,"['*petitioners1_name', '*respondents1_name', '...",MN,CONFIDENTIAL \n\nState of Minnesota \nCounty \...,"[-0.049897458829322214, -0.04619066424846791, ...","[0.314425780819251, -0.13544407857953608, -0.0..."
3,9ff5c8ab7729bc4343ab717c29dba495,Financial Institution Receipt of Letters,NaN,"['ES-02-00-00-00', 'ES-00-00-00-00']",2.0,2.5,8,2023-03,https://supremecourt.nebraska.gov/sites/defaul...,"['fullcountystatement222', 'typeofcourt222', '...",https://courtformsonline.org/forms/9ff5c8ab772...,"['fullcountystatement', 'typeofcourt', 'county...",NE,Nebraska State Court Form \nREQUIRED \nCC 16:2...,"[-0.11940406085949186, -0.09369925525208007, 0...","[-0.36611173224313737, 0.1889065835520422, -0...."
4,9ff5c8ab7729bc4343ab717c29dba495,Financial Institution Receipt of Letters,NaN,"['ES-02-00-00-00', 'ES-00-00-00-00']",2.0,2.5,8,2023-03,https://supremecourt.nebraska.gov/sites/defaul...,"['fullcountystatement222', 'typeofcourt222', '...",https://courtformsonline.org/forms/9ff5c8ab772...,"['fullcountystatement', 'typeofcourt', 'county...",NE,Nebraska State Court Form \nREQUIRED \nCC 16:2...,"[-0.11940406085949186, -0.09369925525208007, 0...","[-0.36611173224313737, 0.1889065835520422, -0...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24714,a12264898b761f4ecdac8977db1116c0,INSTRUCTIONS FOR FILLING OUT,NaN,[],2.0,18.0,7,2023-03,https://supremecourt.nebraska.gov/sites/defaul...,"['c', 'b_enter_your_current_first__middle__and...",https://courtformsonline.org/forms/a12264898b7...,"['c', 'enter_current_first_middle_last', 'a', ...",NE,INSTRUCTIONS FOR COMPLETING THE PETITION FOR N...,"[-0.06006695733339223, -0.028799061440915305, ...","[-0.46913302094324716, -0.19839782704787667, 0..."
24715,a078923719fb98f59f9a95de2ad6bfb0,Home Detention Screening Criteria (for Home De...,NaN,[],4.0,8.25,10,2023-03,https://www.njcourts.gov/sites/default/files/f...,"['cmpltNum', 'res1Addr', 'res6Other', 'res1Pho...",https://courtformsonline.org/forms/a078923719f...,"['cmplt_num', 'res_addr', 'res', 'res_phone', ...",NJ,New Jersey Judiciary \nHome Detention Screenin...,"[-0.07090828418662846, -0.023195665248430655, ...","[-0.45737832229343584, -0.09094739539842009, -..."
24716,a074881877409c04b648a9fa2aee6361,Microsoft Word - D4F99D126C08CAF6FD5BB80FD7CFF...,NaN,['GO-00-00-00-00'],3.0,1.333333333333333,13,2023-03,https://selfhelp.nvcourts.gov/images/spanish/E...,"['undefined', 'Plaintiff', 'undefined_2', 'de ...",https://courtformsonline.org/forms/a0748818774...,"['undefined__1', 'plaintiff', 'undefined__2', ...",NV,DISTRICT COURT \nTRIBUNAL DE DISTR

In [34]:
for state in csv_df["jurisdiction"].unique():
    tmp_df = csv_df[csv_df["jurisdiction"]==state][["id","title","group","pages","reading_level","downloaded_on","original_url","original_fields","new_url","standardized_fields","jurisdiction"]]
    #display(tmp_df)
    tmp_df.to_csv("docs/data/%s_form_data.csv"%(state), index=False, encoding="utf-8")    

tmp_df = csv_df[["id","title","group","list_codes","pages","f_per_p","reading_level","downloaded_on","original_url","original_fields","new_url","standardized_fields","jurisdiction","text"]]
tmp_df.to_csv("../data/form_data.csv", index=False, encoding="utf-8")       
tmp_df.head()

,id,title,group,list_codes,pages,f_per_p,reading_level,downloaded_on,original_url,original_fields,new_url,standardized_fields,jurisdiction,text
0,9fffe321a9fbafdb7964293e8c88e265,Writ of Possession,NaN,['GO-00-00-00-00'],2.0,5.5,13,2023-03,https://www.courts.state.hi.us/docs/form/kauai...,['clerk__district_court_of_the_above_circuit__...,https://courtformsonline.org/forms/9fffe321a9f...,"['clerk_district_court_state', 'duly_authorize...",HI,Writ of Possession \n\n \n\nin the District ...
1,9ffd35f8778e21212dd0f319ebb497d2,Instructions For Florida Supreme Court Approve...,NaN,"['ES-00-00-00-00', 'FA-00-00-00-00']",9.0,7.888889,20,2023-03,https://www.flcourts.gov/content/download/6859...,"['JUDICIAL CIRCUIT', 'COUNTY FLORIDA', 'Case n...",https://courtformsonline.org/forms/9ffd35f8778...,"['judicia_l_circuit', 'count_florida', '*docke...",FL,INSTRUCTIONS FOR FLORIDA SUPREME COURT APPROVE...
2,9ffba05824a302bc69c7fded6244af49,GAL - Fee Waiver Application (Family),NaN,['GO-00-00-00-00'],3.0,12.666667,10,2023-03,https://www.mncourts.gov/mncourtsgov/media/Cou...,"['petitioner', 'respondent', 'if_a_dependent_l...",https://courtformsonline.org/forms/9ffba05824a...,"['*petitioners1_name', '*respondents1_name', '...",MN,CONFIDENTIAL \n\nState of Minnesota \nCounty \...
3,9ff5c8ab7729bc4343ab717c29dba495,Financial Institution Receipt of Letters,NaN,"['ES-02-00-00-00', 'ES-00-00-00-00']",2.0,2.5,8,2023-03,https://supremecourt.nebraska.gov/sites/defaul...,"['fullcountystatement222', 'typeofcourt222', '...",https://courtformsonline.org/forms/9ff5c8ab772...,"['fullcountystatement', 'typeofcourt', 'county...",NE,Nebraska State Court Form \nREQUIRED \nCC 16:2...
4,9ff5c8ab7729bc4343ab717c29dba495,Financial Institution Receipt of Letters,NaN,"['ES-02-00-00-00', 'ES-00-00-00-00']",2.0,2.5,8,2023-03,https://supremecourt.nebraska.gov/sites/defaul...,"['fullcountystatement222', 'typeofcourt222', '...",https://courtformsonline.org/forms/9ff5c8ab772...,"['fullcountystatement', 'typeofcourt', 'county...",NE,Nebraska State Court Form \nREQUIRED \nCC 16:2...


In [36]:
%%time
tmp_df = pd.read_csv("../data/form_data.csv", encoding="utf-8")
tmp_df.head()

CPU times: user 1.81 s, sys: 247 ms, total: 2.05 s
Wall time: 2.07 s


,id,title,group,list_codes,pages,f_per_p,reading_level,downloaded_on,original_url,original_fields,new_url,standardized_fields,jurisdiction,text
0,9fffe321a9fbafdb7964293e8c88e265,Writ of Possession,NaN,['GO-00-00-00-00'],2.0,5.5,13,2023-03,https://www.courts.state.hi.us/docs/form/kauai...,['clerk__district_court_of_the_above_circuit__...,https://courtformsonline.org/forms/9fffe321a9f...,"['clerk_district_court_state', 'duly_authorize...",HI,Writ of Possession \n\n \n\nin the District ...
1,9ffd35f8778e21212dd0f319ebb497d2,Instructions For Florida Supreme Court Approve...,NaN,"['ES-00-00-00-00', 'FA-00-00-00-00']",9.0,7.888888888888889,20,2023-03,https://www.flcourts.gov/content/download/6859...,"['JUDICIAL CIRCUIT', 'COUNTY FLORIDA', 'Case n...",https://courtformsonline.org/forms/9ffd35f8778...,"['judicia_l_circuit', 'count_florida', '*docke...",FL,INSTRUCTIONS FOR FLORIDA SUPREME COURT APPROVE...
2,9ffba05824a302bc69c7fded6244af49,GAL - Fee Waiver Application (Family),NaN,['GO-00-00-00-00'],3.0,12.666666666666666,10,2023-03,https://www.mncourts.gov/mncourtsgov/media/Cou...,"['petitioner', 'respondent', 'if_a_dependent_l...",https://courtformsonline.org/forms/9ffba05824a...,"['*petitioners1_name', '*respondents1_name', '...",MN,CONFIDENTIAL \n\nState of Minnesota \nCounty \...
3,9ff5c8ab7729bc4343ab717c29dba495,Financial Institution Receipt of Letters,NaN,"['ES-02-00-00-00', 'ES-00-00-00-00']",2.0,2.5,8,2023-03,https://supremecourt.nebraska.gov/sites/defaul...,"['fullcountystatement222', 'typeofcourt222', '...",https://courtformsonline.org/forms/9ff5c8ab772...,"['fullcountystatement', 'typeofcourt', 'county...",NE,Nebraska State Court Form \nREQUIRED \nCC 16:2...
4,9ff5c8ab7729bc4343ab717c29dba495,Financial Institution Receipt of Letters,NaN,"['ES-02-00-00-00', 'ES-00-00-00-00']",2.0,2.5,8,2023-03,https://supremecourt.nebraska.gov/sites/defaul...,"['fullcountystatement222', 'typeofcourt222', '...",https://courtformsonline.org/forms/9ff5c8ab772...,"['fullcountystatement', 'typeofcourt', 'county...",NE,Nebraska State Court Form \nREQUIRED \nCC 16:2...


In [366]:
all_jurs = [
        ['AL', 'Alabama'], 
        ['AK', 'Alaska'], 
        ['AZ', 'Arizona'], 
        ['AR', 'Arkansas'], 
        ['CA', 'California'], 
        ['CO', 'Colorado'], 
        ['CT', 'Connecticut'], 
        ['DC', 'District of Columbia'], 
        ['DE', 'Delaware'], 
        ['FL', 'Florida'], 
        ['GA', 'Georgia'], 
        ['HI', 'Hawaii'], 
        ['ID', 'Idaho'], 
        ['IL', 'Illinois'], 
        ['IN', 'Indiana'], 
        ['IA', 'Iowa'], 
        ['KS', 'Kansas'], 
        ['KY', 'Kentucky'], 
        ['LA', 'Louisiana'], 
        ['ME', 'Maine'], 
        ['MD', 'Maryland'], 
        ['MA', 'Massachusetts'], 
        ['MI', 'Michigan'], 
        ['MN', 'Minnesota'], 
        ['MS', 'Mississippi'], 
        ['MO', 'Missouri'], 
        ['MT', 'Montana'], 
        ['NE', 'Nebraska'], 
        ['NV', 'Nevada'], 
        ['NH', 'New Hampshire'], 
        ['NJ', 'New Jersey'], 
        ['NM', 'New Mexico'], 
        ['NY', 'New York'], 
        ['NC', 'North Carolina'], 
        ['ND', 'North Dakota'], 
        ['OH', 'Ohio'], 
        ['OK', 'Oklahoma'], 
        ['OR', 'Oregon'], 
        ['PA', 'Pennsylvania'], 
        ['RI', 'Rhode Island'], 
        ['SC', 'South Carolina'], 
        ['SD', 'South Dakota'], 
        ['TN', 'Tennessee'], 
        ['TX', 'Texas'], 
        ['UT', 'Utah'], 
        ['VT', 'Vermont'], 
        ['VA', 'Virginia'], 
        ['WA', 'Washington'], 
        ['WV', 'West Virginia'], 
        ['WI', 'Wisconsin'], 
        ['WY', 'Wyoming'],
        ['ca1', 'First Circuit']
       ]

default_jur = "MA"

jurs = []
for jur in all_jurs:
    if jur[0] in forms_df["jurisdiction"].unique():
    #if jur[0] in tmp_df["jurisdiction"].unique():
        jurs.append(jur)
        
print("length",len(jurs))

length 47


In [38]:
from sklearn.preprocessing import normalize
import spacy
import numpy as np
import json

In [39]:
#if use_model==0:
import en_core_web_lg
nlp = en_core_web_lg.load() #nlp = spacy.load('en_core_web_lg')

/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/spacy/util.py:837: UserWarning: [W095] Model 'en_core_web_lg' (3.5.0) was trained with spaCy v3.5 and may not be 100% compatible with the current version (3.3.1). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [40]:
def norm(row):
    """Normalize a word vector."""
    try:
        matrix = row.reshape(1, -1).astype(np.float64)
        return normalize(matrix, axis=1, norm="l2")[0]
    except Exception as e:
        print("===================")
        print("Error: ", e)
        print("===================")
        return np.NaN

def vectorize_one(text,mode):
    """Vectorize a string of text."""
    
    if mode==0:
        return norm(nlp(str(text)).vector)
    #elif mode==1:
    #    #return norm(np.average(nlp2(str(text))._.trf_data.tensors[-1], axis=0))
    #    return norm(np.average(normalize(nlp2(str(text))._.trf_data.tensors[-1], axis=1, norm="l2"), axis=0))
    elif mode ==2:
        return np.array(get_embedding(text,mode=mode))
    else:
        return []
    
def vectorize(text,mode):
    """Vectorize a string (or strings) of text."""
    
    if isinstance(text, str):
        output = vectorize_one(text,mode)
    else:
        output = []
        for item in text:
            output.append(vectorize_one(item,mode).tolist())
                
    return list(output)

In [41]:
vectorize("text goes here",0)

[0.08158453646136106,
 0.007466353416808095,
 0.016883352096717128,
 0.004566527569989328,
 0.050255235812826975,
 -0.04424023811869246,
 0.087127284055433,
 0.032124483112883025,
 -0.07219764455542918,
 0.04688190383876885,
 0.18424665477074786,
 0.06636156767070442,
 -0.13134308988673912,
 0.05714450579038107,
 0.06494501723881306,
 -0.00585148165307303,
 0.11972666713514618,
 0.024047732724976303,
 -0.002279476965811508,
 0.04741670626822111,
 -0.039431142566836844,
 -0.049916217274367514,
 0.01114373181298966,
 -0.0885908938000656,
 -0.09992365830760278,
 -0.03596910775339061,
 -0.024571447299505052,
 -0.08243996632232739,
 -0.018387797835605745,
 0.058310202168273845,
 -0.027437020314078454,
 -0.058864572348674135,
 -0.029197199149973867,
 -0.08410548817781312,
 -0.04098165311250829,
 -0.06260622290869856,
 -0.0609336605312902,
 -0.03625142816933378,
 0.12316732893561295,
 0.08256255650899155,
 -0.05228367336291345,
 0.02452639795261491,
 -0.04484499445186336,
 -0.0013908705794940

In [31]:
import requests
import json

def vectorize(text):
    governor(0,50,66.66,0)
    try:
        # This is your real token.
        token = ""
        headers = { "Authorization": "Bearer " + token, "Content-Type":"application/json" }
        # Find Embeddings
        body = {
          "text": text
        }
        r = requests.post('https://tools.suffolklitlab.org/vectorize/', headers=headers, data=json.dumps(body))
        output_1 = r.json()

        return output_1["embeddings"]
    except:
        print("ERROR!!!")
        return []

In [32]:
vectorize("text goes here")

[0.08158453646136106,
 0.007466353416808095,
 0.016883352096717128,
 0.004566527569989328,
 0.050255235812826975,
 -0.04424023811869246,
 0.087127284055433,
 0.032124483112883025,
 -0.07219764455542918,
 0.04688190383876885,
 0.18424665477074786,
 0.06636156767070442,
 -0.13134308988673912,
 0.05714450579038107,
 0.06494501723881306,
 -0.00585148165307303,
 0.11972666713514618,
 0.024047732724976303,
 -0.002279476965811508,
 0.04741670626822111,
 -0.039431142566836844,
 -0.049916217274367514,
 0.01114373181298966,
 -0.0885908938000656,
 -0.09992365830760278,
 -0.03596910775339061,
 -0.024571447299505052,
 -0.08243996632232739,
 -0.018387797835605745,
 0.058310202168273845,
 -0.027437020314078454,
 -0.058864572348674135,
 -0.029197199149973867,
 -0.08410548817781312,
 -0.04098165311250829,
 -0.06260622290869856,
 -0.0609336605312902,
 -0.03625142816933378,
 0.12316732893561295,
 0.08256255650899155,
 -0.05228367336291345,
 0.02452639795261491,
 -0.04484499445186336,
 -0.0013908705794940

In [42]:
from sklearn.decomposition import PCA
import re
import json

In [43]:
%%time
test_vec = tmp_df.copy()
#test_vec["vec"] = test_vec["text"].apply(vectorize)

CPU times: user 3.09 ms, sys: 1.46 ms, total: 4.55 ms
Wall time: 3.34 ms


In [311]:
test_vec = files_df[files_df["jurisdiction"]=="ca1"]
test_vec

,id,jurisdiction,source,title,group,url,filename,downloaded,pages,fields,fields_conf,fields_old,f_per_p,reading,list,text,downloaded_on
24719,4cd34891da97f2dc9271a3559d74e2a7,ca1,ca1.uscourts.gov,Bill of Costs,NaN,javascript:alert('not available');,ao133_Bill_of_Cost.pdf,2023-04,2.0,"['district_information', 'clerk_sig', 'clerk_d...","[0.45, 0.41000000000000003, 0.4, 1.0, 1.0, 0.3...","['District Information', 'Clerk.Sig', 'Clerk.D...",38.0,10,[],",\n\n \n\nUNITED STATES DISTRICT COURT\nfor th...",2023-04
24720,9e839dde6965d468952dd00bdda7ba32,ca1,ca1.uscourts.gov,U.S. Marshals Service Form 285 - Service of Pr...,NaN,javascript:alert('not available');,USM 285 Process Receipt and Return.pdf,2023-04,2.0,"['plaintiff', 'courtcase', 'defendant', 'proce...","[1.0, 0.35000000000000003, 1.0, 0.33, 1.0, 1.0...","['Plaintiff', 'Courtcase', 'Defendant', 'Proce...",18.5,7,[],USM-285 is a 5-part form. Fill out the form a...,2023-04
24721,fa74e5c8651cc1c537c653f288c00b36,ca1,ca1.uscourts.gov,U.S. Marshals Service Form 285 - Service of Pr...,NaN,javascript:alert('not available');,US Marshal Form 285 Process Receipt and Return...,2023-04,2.0,"['plaintiff', 'courtcase', 'defendant', 'proce...","[1.0, 0.35000000000000003, 1.0, 0.33, 1.0, 1.0...","['Plaintiff', 'Courtcase', 'Defendant', 'Proce...",18.5,7,[],USM-285 is a 5-part form. Fill out the form a...,2023-04
24722,b9bdd37aeb287a1cbd8372d3c365234e,ca1,ca1.uscourts.gov,Step by Step Guide to Filing a Civil Action Pr...,NaN,javascript:alert('not available');,Step by Step Guide to Filing a Civil Action Pr...,2023-04,5.0,"['filing_civil_states_court', 'u_c', 'must_cal...","[0.51, 0.29, 0.4, 0.45, 0.34, 0.4, 0.33, 0.44,...",['a_simple_guide_to_filing_a_civil_action_in_t...,8.0,9,"['CO-00-00-00-00', 'CO-07-00-00-00']","4/3/23, 3:14 PM\n\nStep by Step Guide to Filin...",2023-04
24723,8e4aa59b8ba99531160331cd1ae43709,ca1,ca1.uscourts.gov,SS_Complaint,NaN,javascript:alert('not available');,SS_Complaint.pdf,2023-04,2.0,"['plaintiff', 'civil', 'commissioner_social_se...","[1.0, 0.44, 0.41000000000000003, 1.0, 0.39, 1....","['plaintiff', 'civil_no', 'commissioner_of_soc...",8.0,14,['GO-00-00-00-00'],COMPLAINT \n \n\n \n\nThe above-named Plaint...,2023-04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24815,8d658185fad20e95719f76b30db16e1f,ca1,ca1.uscourts.gov,2254 Instructions to Inmates (English Spanish),NaN,javascript:alert('not available');,2254 Instructions to Inmates (English Spanish)...,2023-04,2.0,"['page_check__1', 'page_check__2', 'page_check...","[0.51, 0.33, 0.33, 0.33, 0.33, 0.33, 0.33, 0.2...","['page_0_check_0', 'page_0_check_1', 'page_0_c...",5.5,0,[],,2023-04
24816,2d6eebf019d539ab3c24009724e7c079,ca1,ca1.uscourts.gov,1983 Notice to Inmates (English Spanish),NaN,javascript:alert('not available');,1983 Notice to Inmates (English Spanish).pdf,2023-04,4.0,"['page_check__1', 'page_check__2', 'page_check...","[0.51, 0.33, 0.33, 0.33, 0.26]","['page_0_check_0', 'page_0_check_1', 'page_0_c...",1.25,0,[],,2023-04
24817,0090296bf59dd7937947ce57a8f1f0e3,ca1,ca1.uscourts.gov,1983 Instructions to Inmates (English Spanish),NaN,javascript:alert('not available');,1983 Instructions to Inmates (English Spanish)...,2023-04,2.0,"['page_check__1', 'page_check__2', 'page_check...","[0.51, 0.33, 0.33, 0.33, 0.33, 0.33, 0.26, 0.2...","['page_0_check_0', 'page_0_check_1', 'page_0_c...",6.5,0,[],,2023-04
24818,4181fb4aaedd78a911d19e518293af6c,ca1,ca1.uscourts.gov,1331 Instructions to Inmates (English),NaN,javascript:alert('not available');,1331 Instructions to Inmates (English).pdf,2023-04,3.0,[],[],[],0.0,0,[],,2023-04


In [312]:
%%time
test_vec["vec"] = "[]"

#test_vec = test_vec[:2]
#display(test_vec)

i = 0
p_1 = round(len(test_vec)/100)
print("Starting...")
for index,row in test_vec.iterrows():
    if (row["vec"]=="[]") & (len(str(row["text"]))>5):
        try:
            embedding = vectorize(row["text"],0)
            test_vec.loc[index,"vec"] = str(embedding)
        except:
            print("ERROR!!!")
    i+=1
    if i%p_1==0:
        print("{}) {}% done...".format(i,i/p_1))
        governor(0,50,100,1)
    else:
        governor(0,50,100,0)
print("DONE\n\n")

<timed exec>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


Starting...


/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/pandas/core/indexing.py:1817: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(loc, value, pi)
/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/pandas/core/indexing.py:1817: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(loc, value, pi)


1) 1.0% done...

 Usage: CPU 7.9% & Memory 65.7%

2) 2.0% done...

 Usage: CPU 15.9% & Memory 65.7%



/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/pandas/core/indexing.py:1817: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(loc, value, pi)
/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/pandas/core/indexing.py:1817: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(loc, value, pi)
/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/pandas/core/indexing.py:1817: SettingWithCopyWarning: 
A value is trying to be s

...

/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/pandas/core/indexing.py:1817: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(loc, value, pi)


93) 93.0% done...

 Usage: CPU 16.6% & Memory 67.3%



/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/pandas/core/indexing.py:1817: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(loc, value, pi)
/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/pandas/core/indexing.py:1817: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(loc, value, pi)


94) 94.0% done...

 Usage: CPU 15.9% & Memory 67.3%

95) 95.0% done...

 Usage: CPU 16.0% & Memory 67.3%

96) 96.0% done...

 Usage: CPU 0.0% & Memory 67.3%

97) 97.0% done...

 Usage: CPU 0.0% & Memory 67.3%

98) 98.0% done...

 Usage: CPU 0.0% & Memory 67.3%

99) 99.0% done...

 Usage: CPU 0.0% & Memory 67.3%

100) 100.0% done...

 Usage: CPU 0.0% & Memory 67.3%

101) 101.0% done...

 Usage: CPU 0.0% & Memory 67.3%

DONE


CPU times: user 14.9 s, sys: 1.03 s, total: 15.9 s
Wall time: 16 s


/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/pandas/core/indexing.py:1817: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(loc, value, pi)
/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/pandas/core/indexing.py:1817: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(loc, value, pi)


In [313]:
test_vec

,id,jurisdiction,source,title,group,url,filename,downloaded,pages,fields,fields_conf,fields_old,f_per_p,reading,list,text,downloaded_on,vec
24719,4cd34891da97f2dc9271a3559d74e2a7,ca1,ca1.uscourts.gov,Bill of Costs,NaN,javascript:alert('not available');,ao133_Bill_of_Cost.pdf,2023-04,2.0,"['district_information', 'clerk_sig', 'clerk_d...","[0.45, 0.41000000000000003, 0.4, 1.0, 1.0, 0.3...","['District Information', 'Clerk.Sig', 'Clerk.D...",38.0,10,[],",\n\n \n\nUNITED STATES DISTRICT COURT\nfor th...",2023-04,"[-0.04831155256418288, -0.07740390560441368, -..."
24720,9e839dde6965d468952dd00bdda7ba32,ca1,ca1.uscourts.gov,U.S. Marshals Service Form 285 - Service of Pr...,NaN,javascript:alert('not available');,USM 285 Process Receipt and Return.pdf,2023-04,2.0,"['plaintiff', 'courtcase', 'defendant', 'proce...","[1.0, 0.35000000000000003, 1.0, 0.33, 1.0, 1.0...","['Plaintiff', 'Courtcase', 'Defendant', 'Proce...",18.5,7,[],USM-285 is a 5-part form. Fill out the form a...,2023-04,"[-0.09707063391750333, -0.05320557885062505, -..."
24721,fa74e5c8651cc1c537c653f288c00b36,ca1,ca1.uscourts.gov,U.S. Marshals Service Form 285 - Service of Pr...,NaN,javascript:alert('not available');,US Marshal Form 285 Process Receipt and Return...,2023-04,2.0,"['plaintiff', 'courtcase', 'defendant', 'proce...","[1.0, 0.35000000000000003, 1.0, 0.33, 1.0, 1.0...","['Plaintiff', 'Courtcase', 'Defendant', 'Proce...",18.5,7,[],USM-285 is a 5-part form. Fill out the form a...,2023-04,"[-0.09707063391750333, -0.05320557885062505, -..."
24722,b9bdd37aeb287a1cbd8372d3c365234e,ca1,ca1.uscourts.gov,Step by Step Guide to Filing a Civil Action Pr...,NaN,javascript:alert('not available');,Step by Step Guide to Filing a Civil Action Pr...,2023-04,5.0,"['filing_civil_states_court', 'u_c', 'must_cal...","[0.51, 0.29, 0.4, 0.45, 0.34, 0.4, 0.33, 0.44,...",['a_simple_guide_to_filing_a_civil_action_in_t...,8.0,9,"['CO-00-00-00-00', 'CO-07-00-00-00']","4/3/23, 3:14 PM\n\nStep by Step Guide to Filin...",2023-04,"[-0.08711550572512726, -0.01678067318771126, -..."
24723,8e4aa59b8ba99531160331cd1ae43709,ca1,ca1.uscourts.gov,SS_Complaint,NaN,javascript:alert('not available');,SS_Complaint.pdf,2023-04,2.0,"['plaintiff', 'civil', 'commissioner_social_se...","[1.0, 0.44, 0.41000000000000003, 1.0, 0.39, 1....","['plaintiff', 'civil_no', 'commissioner_of_soc...",8.0,14,['GO-00-00-00-00'],COMPLAINT \n \n\n \n\nThe above-named Plaint...,2023-04,"[-0.036852800611890515, -0.01694938332420521, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24815,8d658185fad20e95719f76b30db16e1f,ca1,ca1.uscourts.gov,2254 Instructions to Inmates (English Spanish),NaN,javascript:alert('not available');,2254 Instructions to Inmates (English Spanish)...,2023-04,2.0,"['page_check__1', 'page_check__2', 'page_check...","[0.51, 0.33, 0.33, 0.33, 0.33, 0.33, 0.33, 0.2...","['page_0_check_0', 'page_0_check_1', 'page_0_c...",5.5,0,[],,2023-04,[]
24816,2d6eebf019d539ab3c24009724e7c079,ca1,ca1.uscourts.gov,1983 Notice to Inmates (English Spanish),NaN,javascript:alert('not available');,1983 Notice to Inmates (English Spanish).pdf,2023-04,4.0,"['page_check__1', 'page_check__2', 'page_check...","[0.51, 0.33, 0.33, 0.33, 0.26]","['page_0_check_0', 'page_0_check_1', 'page_0_c...",1.25,0,[],,2023-04,[]
24817,0090296bf59dd7937947ce57a8f1f0e3,ca1,ca1.uscourts.gov,1983 Instructions to Inmates (English Spanish),NaN,javascript:alert('not available');,1983 Instructions to Inmates (English Spanish)...,2023-04,2.0,"['page_check__1', 'page_check__2', 'page_check...","[0.51, 0.33, 0.33, 0.33, 0.33, 0.33, 0.26, 0.2...","['page_0_check_0', 'page_0_check_1', 'page_0_c...",6.5,0,[],,2023-04,[]
24818,4181fb4aaedd78a911d19e518293af6c,ca1,ca1.uscourts.gov,1331 Instructions to Inmates (English),NaN,javascript:alert('not available');,1331 Instructions to Inmates (English).pdf,2023-04,3.0,[],[],[],0.0,0,[],,2023-04,[]


In [314]:
hold_this = test_vec.copy()

In [315]:
test_vec = hold_this.copy()

In [316]:
from ast import literal_eval

In [317]:
test_vec = test_vec[test_vec["vec"]!="[]"]

In [318]:
test_vec["vec"] = test_vec["vec"].apply(literal_eval)

In [319]:
test_vec["vec"] = test_vec["vec"].apply(np.array)

In [320]:
vec_df_low = test_vec["vec"].to_list()

In [321]:
vec_df_low

[array([-4.83115526e-02, -7.74039056e-02, -5.63995740e-02, -3.35956475e-02,
         2.12275368e-01,  4.85214941e-02, -1.26646572e-03,  7.37926256e-02,
         2.19138541e-02, -6.49616763e-02,  1.65744705e-01,  5.48276175e-02,
        -6.23653562e-02,  8.72427647e-02,  2.38892094e-02,  3.81506159e-02,
        -1.81866240e-02, -3.74441625e-02, -7.02034555e-02, -1.47298080e-01,
         3.79403305e-02, -1.75527322e-02, -3.44024415e-02,  4.30541233e-03,
        -5.55616403e-02, -4.21683324e-02, -3.25029291e-02, -5.93641976e-02,
         2.46998062e-02,  6.98652564e-02,  9.66963393e-03, -7.97794220e-02,
        -7.33235357e-02, -7.25924524e-02, -9.90728928e-02,  8.60889685e-03,
        -2.92533729e-03,  2.79662300e-02,  3.22899235e-02,  9.09066141e-02,
         6.40650639e-02, -9.95682674e-03, -3.06704509e-02, -2.08920728e-02,
        -9.52685785e-02,  4.86228246e-02,  6.46877931e-03, -3.28484540e-02,
        -2.27262398e-02, -3.30284158e-02, -4.95280437e-02,  7.40184921e-02,
        -9.9

In [203]:
pca2 = PCA(n_components=100)

In [204]:
%%time
pca2.fit(vec_df_low)

CPU times: user 13.5 s, sys: 1.72 s, total: 15.3 s
Wall time: 18.1 s


PCA(n_components=100)

In [205]:
%%time
dump(pca2,"../data/pca.joblib")

CPU times: user 707 ms, sys: 7.59 ms, total: 715 ms
Wall time: 217 ms


['../data/pca.joblib']

In [732]:
def pca_form(text):
    return list(pca2.transform([vectorize(text)])[0])

In [734]:
#pca_form("test")

In [322]:
vec_df_low = pca2.transform(vec_df_low)
test_vec["PCA"] = list(vec_df_low)
test_vec

,id,jurisdiction,source,title,group,url,filename,downloaded,pages,fields,fields_conf,fields_old,f_per_p,reading,list,text,downloaded_on,vec,PCA
24719,4cd34891da97f2dc9271a3559d74e2a7,ca1,ca1.uscourts.gov,Bill of Costs,NaN,javascript:alert('not available');,ao133_Bill_of_Cost.pdf,2023-04,2.0,"['district_information', 'clerk_sig', 'clerk_d...","[0.45, 0.41000000000000003, 0.4, 1.0, 1.0, 0.3...","['District Information', 'Clerk.Sig', 'Clerk.D...",38.0,10,[],",\n\n \n\nUNITED STATES DISTRICT COURT\nfor th...",2023-04,"[-0.04831155256418288, -0.07740390560441368, -...","[-0.38332373439287265, -0.11166643156896179, 0..."
24720,9e839dde6965d468952dd00bdda7ba32,ca1,ca1.uscourts.gov,U.S. Marshals Service Form 285 - Service of Pr...,NaN,javascript:alert('not available');,USM 285 Process Receipt and Return.pdf,2023-04,2.0,"['plaintiff', 'courtcase', 'defendant', 'proce...","[1.0, 0.35000000000000003, 1.0, 0.33, 1.0, 1.0...","['Plaintiff', 'Courtcase', 'Defendant', 'Proce...",18.5,7,[],USM-285 is a 5-part form. Fill out the form a...,2023-04,"[-0.09707063391750333, -0.05320557885062505, -...","[-0.4660632554259611, -0.08091559578805463, -0..."
24721,fa74e5c8651cc1c537c653f288c00b36,ca1,ca1.uscourts.gov,U.S. Marshals Service Form 285 - Service of Pr...,NaN,javascript:alert('not available');,US Marshal Form 285 Process Receipt and Return...,2023-04,2.0,"['plaintiff', 'courtcase', 'defendant', 'proce...","[1.0, 0.35000000000000003, 1.0, 0.33, 1.0, 1.0...","['Plaintiff', 'Courtcase', 'Defendant', 'Proce...",18.5,7,[],USM-285 is a 5-part form. Fill out the form a...,2023-04,"[-0.09707063391750333, -0.05320557885062505, -...","[-0.4660632554259611, -0.08091559578805463, -0..."
24722,b9bdd37aeb287a1cbd8372d3c365234e,ca1,ca1.uscourts.gov,Step by Step Guide to Filing a Civil Action Pr...,NaN,javascript:alert('not available');,Step by Step Guide to Filing a Civil Action Pr...,2023-04,5.0,"['filing_civil_states_court', 'u_c', 'must_cal...","[0.51, 0.29, 0.4, 0.45, 0.34, 0.4, 0.33, 0.44,...",['a_simple_guide_to_filing_a_civil_action_in_t...,8.0,9,"['CO-00-00-00-00', 'CO-07-00-00-00']","4/3/23, 3:14 PM\n\nStep by Step Guide to Filin...",2023-04,"[-0.08711550572512726, -0.01678067318771126, -...","[-0.5013517536917323, -0.20419104337568633, 0...."
24723,8e4aa59b8ba99531160331cd1ae43709,ca1,ca1.uscourts.gov,SS_Complaint,NaN,javascript:alert('not available');,SS_Complaint.pdf,2023-04,2.0,"['plaintiff', 'civil', 'commissioner_social_se...","[1.0, 0.44, 0.41000000000000003, 1.0, 0.39, 1....","['plaintiff', 'civil_no', 'commissioner_of_soc...",8.0,14,['GO-00-00-00-00'],COMPLAINT \n \n\n \n\nThe above-named Plaint...,2023-04,"[-0.036852800611890515, -0.01694938332420521, ...","[0.6413162562535697, -0.08093859082756127, -0...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24811,d7ea2ec66e81593b12cf20e44df27f91,ca1,ca1.uscourts.gov,Petition for a Writ of Habeas Corpus Under 28 ...,NaN,javascript:alert('not available');,AO 242 Petition for Writ of Habeas Corpus 28 2...,2023-04,9.0,"['copies_clerk_united_states', 'district', 'fu...","[0.64, 0.24, 0.41000000000000003, 1.0, 0.42, 1...","['copies to the clerk of the United States', '...",21.333333,9,"['CO-00-00-00-00', 'CO-07-00-00-00']",AO 242 (Rev. 09/17) Petition for a Writ of Ha...,2023-04,"[-0.10216455927838833, -0.06573610288587095, 0...","[-0.3624968052127185, 0.12664702370800085, -0...."
24812,02253cfdd0a9972edab1f39701faa468,ca1,ca1.uscourts.gov,Petition for Relief from a Conviction or Sentence,NaN,javascript:alert('not available');,AO 241 Petition for Habeas Corpus 28 2254.pdf,2023-04,16.0,"['excess_amt', 'return_address', 'pages', 'cou...","[0.6000000000000001, 0.34, 0.27, 0.51, 0.49, 0...","['Excess.Amt', 'Return Address', 'Pages', 'Cou...",22.0625,9,"['CO-00-00-00-00', 'CO-07-00-00-00']",AO 241\n(Rev. 01/15)\n\nPage 1\n\nPetition ...,2023-04,"[-0.09588213628942653, -0.04376181782310023, 0...","[-0.40681978548062475, 0.04517782487526555, -0..."
24813,466d05c03ae300209e

In [326]:
test_vec.columns

Index(['id', 'jurisdiction', 'source', 'title', 'group', 'url', 'filename',
       'downloaded', 'pages', 'fields', 'fields_conf', 'fields_old', 'f_per_p',
       'reading', 'list', 'text', 'downloaded_on', 'vec', 'PCA'],
      dtype='object')

In [327]:
test_vec = test_vec.rename(columns={
                                'list':'list_codes'
                                   })

In [325]:
test_vec_0.columns

Index(['id', 'title', 'group', 'list_codes', 'pages', 'f_per_p',
       'reading_level', 'downloaded_on', 'original_url', 'original_fields',
       'new_url', 'standardized_fields', 'jurisdiction', 'text', 'vec', 'PCA'],
      dtype='object')

In [328]:
test_vec.loc[pd.isnull(test_vec["list_codes"]),"list_codes"] = "[]"
test_vec.loc[test_vec["list_codes"]=="{'status': 400, 'message': '`text` must be greater than 5 and less than 5,000 characters long.'}","list_codes"] = "[]"
#test_vec.loc[pd.isnull(test_vec["reading_level"]),"reading_level"] = None

In [333]:
test_vec = pd.concat([test_vec_0,test_vec]) 
test_vec

,id,title,group,list_codes,pages,f_per_p,reading_level,downloaded_on,original_url,original_fields,...,vec,PCA,source,url,filename,downloaded,fields,fields_conf,fields_old,reading
0,9fffe321a9fbafdb7964293e8c88e265,Writ of Possession,NaN,['GO-00-00-00-00'],2.0,5.5,13,2023-03,https://www.courts.state.hi.us/docs/form/kauai...,['clerk__district_court_of_the_above_circuit__...,...,"[-0.027059428771668746, -0.01102143584139791, ...","[0.7096992402661475, -0.1001827335832918, 0.00...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,9ffd35f8778e21212dd0f319ebb497d2,Instructions For Florida Supreme Court Approve...,NaN,"['ES-00-00-00-00', 'FA-00-00-00-00']",9.0,7.888888888888889,20,2023-03,https://www.flcourts.gov/content/download/6859...,"['JUDICIAL CIRCUIT', 'COUNTY FLORIDA', 'Case n...",...,"[-0.08607775506534653, -0.020282120776711276, ...","[-0.4827016710945098, -0.16228668440049665, 0....",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9ffba05824a302bc69c7fded6244af49,GAL - Fee Waiver Application (Family),NaN,['GO-00-00-00-00'],3.0,12.666666666666666,10,2023-03,https://www.mncourts.gov/mncourtsgov/media/Cou...,"['petitioner', 'respondent', 'if_a_dependent_l...",...,"[-0.049897458829322214, -0.04619066424846791, ...","[0.314425780819251, -0.13544407857953608, -0.0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9ff5c8ab7729bc4343ab717c29dba495,Financial Institution Receipt of Letters,NaN,"['ES-02-00-00-00', 'ES-00-00-00-00']",2.0,2.5,8,2023-03,https://supremecourt.nebraska.gov/sites/defaul...,"['fullcountystatement222', 'typeofcourt222', '...",...,"[-0.11940406085949186, -0.09369925525208007, 0...","[-0.36611173224313737, 0.1889065835520422, -0....",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9ff5c8ab7729bc4343ab717c29dba495,Financial Institution Receipt of Letters,NaN,"['ES-02-00-00-00', 'ES-00-00-00-00']",2.0,2.5,8,2023-03,https://supremecourt.nebraska.gov/sites/defaul...,"['fullcountystatement222', 'typeofcourt222', '...",...,"[-0.11940406085949186, -0.09369925525208007, 0...","[-0.36611173224313737, 0.1889065835520422, -0....",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24811,d7ea2ec66e81593b12cf20e44df27f91,Petition for a Writ of Habeas Corpus Under 28 ...,NaN,"['CO-00-00-00-00', 'CO-07-00-00-00']",9.0,21.333333,NaN,2023-04,NaN,NaN,...,"[-0.10216455927838833, -0.06573610288587095, 0...","[-0.3624968052127185, 0.12664702370800085, -0....",ca1.uscourts.gov,javascript:alert('not available');,AO 242 Petition for Writ of Habeas Corpus 28 2...,2023-04,"['copies_clerk_united_states', 'district', 'fu...","[0.64, 0.24, 0.41000000000000003, 1.0, 0.42, 1...","['copies to the clerk of the United States', '...",9
24812,02253cfdd0a9972edab1f39701faa468,Petition for Relief from a Conviction or Sentence,NaN,"['CO-00-00-00-00', 'CO-07-00-00-00']",16.0,22.0625,NaN,2023-04,NaN,NaN,...,"[-0.09588213628942653, -0.04376181782310023, 0...","[-0.40681978548062475, 0.04517782487526555, -0...",ca1.uscourts.gov,javascript:alert('not available');,AO 241 Petition for Habeas Corpus 28 2254.pdf,2023-04,"['excess_amt', 'return_address', 'pages', 'cou...","[0.6000000000000001, 0.34, 0.27, 0.51, 0.49, 0...","['Excess.Amt', 'Return Address', 'Pages', 'Cou...",9
24813,466d05c03ae300209e71215389088593,Application to Proceed in District Court Witho...,NaN,[],2.0,9.0,NaN,2023-04,NaN,NaN,...,"[-0.08570032998686082, -0.05858612555395104, -...","[-0.3679367602784062, -0.1216409522001487, 0.0...",ca1.uscourts.gov,javascript:alert('not available');,AO 240 Application to Proceed in District Cour...,2023-04,"['district_information', 'amount_source_income...","[0.45, 0.42, 1.0, 1.0, 0.33, 0.42, 0.39, 0.42,...","['District Information', 'Amount and source of...",13
24814,d2fefeb0dfa88429249849d83cd9ee71,Application to proceed in District Courts With...,NaN,['MO-00-00-00-00'],5.0,36.6,NaN,2023-04,NaN,NaN,...,"[-0.08851526992102683, -0.06511906793356992, -...","[-0.22340526586032505, 0.16605635253780313, 0....",ca1.uscourts.gov,javascript:alert('

In [341]:
tmp_df = test_vec[["id","title","group","list_codes","pages","f_per_p","reading_level","downloaded_on","original_url","original_fields","new_url","standardized_fields","jurisdiction","text"]]


In [209]:
#test_vec["group"]#.unique()

In [336]:
%%time
formsinfo = "var FormsInfo = [\n\t"
i = 1
for index,row in test_vec.iterrows():
    
    title = row["title"] + " (%s)"%row["jurisdiction"]
    title = re.sub('\s+',' ',title)
    text = row["text"]
    if pd.isnull(text):
        text = ""
    text = title +" "+ text
    reading = row["reading_level"]
    if pd.isnull(reading):
        reading = "null"
    pages = row["pages"]
    if pd.isnull(pages):
        pages = "null"
    f_per_p = row["f_per_p"]
    if pd.isnull(f_per_p):
        f_per_p = "null"
    fields = row["standardized_fields"]
    if pd.isnull(fields):
        fields = []
    formsinfo += "{"
    formsinfo += """
\t"fid":"%s",
\t"jur":"%s",
\t"name":%s,
\t"reading":%s,
\t"pages":%s,
\t"f_per_p":%s,
\t"label":%s,
\t"list":%s,
\t"fields":%s,
\t"text":%s,
\t"vec":%s
"""%(row["id"],row["jurisdiction"],json.dumps(title),reading,pages,f_per_p,json.dumps(row["group"]),row["list_codes"],fields,json.dumps(text),np.array2string(row["PCA"],separator=','))
    formsinfo += "\t}"
    if i < len(test_vec):
        formsinfo += ","
    else:
        formsinfo += "\n"
    i+=1

formsinfo = re.sub("\n|\t","",formsinfo)
    
formsinfo += "]\n"

CPU times: user 20.7 s, sys: 1.09 s, total: 21.8 s
Wall time: 21.8 s


In [337]:
%%time
text_file = open("../data/formsinfo.js", "w")
text_file.write(formsinfo)
text_file.close()

CPU times: user 212 ms, sys: 353 ms, total: 565 ms
Wall time: 1.16 s


In [342]:
def header(title,description,path="./",wide=0):
    
    if wide==1:
        pclass = "content_wide"
    else:
        pclass = "content"
    
    html = """<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">
<HTML xmlns="http://www.w3.org/1999/xhtml"
      xmlns:og="http://ogp.me/ns#"
      xmlns:fb="http://www.facebook.com/2008/fbml">
<HEAD>
    <title>%s</title>
    <!-- Global site tag (gtag.js) - Google Analytics -->
    <script async src="https://www.googletagmanager.com/gtag/js?id=UA-108858221-1"></script>
    <script>
      window.dataLayer = window.dataLayer || [];
      function gtag(){dataLayer.push(arguments);}
      gtag('js', new Date());

      gtag('config', 'UA-108858221-1');
    </script>
    <meta http-equiv="Content-type" content="text/html;charset=UTF-8"/>
    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=0" />
    <meta name="apple-mobile-web-app-capable" content="no" />
    <link rel="apple-touch-icon" href="%simages/icon_300.png"/>
    <meta property="og:type" content="website"/>
    <meta property="og:title" content="%s"/>
    <meta property="og:description" content="%s"/>
    <meta property="og:image" content="%simages/bigdata.png"/>

    <meta name="twitter:card" content="summary_large_image">
    <meta name="twitter:site" content="@SuffolkLITLab">
    <meta name="twitter:creator" content="@SuffolkLITLab">
    <meta name="twitter:title" content="%s">
    <meta name="twitter:description" content="%s">
    <meta name="twitter:image" content="http://www.davidcolarusso.com/images/bigdata.png"/>

    <meta http-equiv="X-UA-Compatible" content="IE=edge" />
    <link rel="shortcut icon" type="image/x-icon" href="/favicon.ico">
    <link rel="apple-touch-icon" href="%simages/icon.png"/>
    <link rel="stylesheet" href="https://code.jquery.com/ui/1.11.1/themes/smoothness/jquery-ui.css">
    <link rel="stylesheet" type="text/css" href="%scss/style.css?v=%s">
    <script src="https://code.jquery.com/jquery-1.11.1.min.js"></script>
    <script src="https://code.jquery.com/jquery-1.10.2.js"></script>
    <script src="https://code.jquery.com/ui/1.11.1/jquery-ui.js"></script>

</HEAD>
<BODY BGCOLOR="#ffffff" BACKGROUND="" MARGINWIDTH="0" MARGINHEIGHT="0">
<div style="box-sizing: border-box;width:100%%;background:#f6d91b;color:black;padding:15px;text-align:center;">
<b>This tool is in beta.</b> <b>CONTENT IS SUBJECT TO CHANGE!</b></div>
<div class="%s">

    <div id="icon" style="background-size: 100px 100px;background-image: url('%simages/seal.jpg');"><a href="%s" alt="home"><img src="%simages/space.gif" width="100px" height="100px;" border="0" alt="LIT Logo"/></a></div>
    <h1 style="text-align:center;">
        The Legal Innovation & Technology Lab's Form Explorer <sup><font size=+1>Beta</font></sup>
        <center style="margin-top:5px;">
          <span class="subtitle">@ Suffolk Law School&nbsp;<font style="font-weight:normal">Last updated %s</font></span>
        </center>
    </h1>
"""%(title,path,title,description,path,title,description,path,path,today,pclass,path,path,path,today)
    return html
    
def footer(path="./"):
    html = """

<div class="footer">
    <a href="/" alt="home"><img src="%simages/blue_logo.png" width="50px" align="left" border="0" alt="LIT Logo"/></a>
<font size=-1><a href="mailto:litlab@suffolk.edu">Email</a> | <a href="https://github.com/SuffolkLITLab" target="_blank">GitHub</a> | <a href="/terms">Terms &amp; Privacy</a> | <a href="/credits">Credits</a></font>
</div>

</BODY>
</HTML>
"""%(path)
    
    return html

# Home page

In [343]:
def home():
    html = """
    <div class="menu_bar">
        <p style="text-align:center;">
      <a href="https://suffolklitlab.org/" class="menu">&nbsp;LIT Lab Home&nbsp;</a>&nbsp;<font style="color:#888;">|</font>&nbsp;
      <a href="." class="menu" style="color:black;">&nbsp;About&nbsp;The&nbsp;Explorer&nbsp;</a>&nbsp;<font style="color:#888;">|</font>&nbsp;
      <a href="compare" class="menu">Find&nbsp;&amp;&nbsp;Compare&nbsp;</a><font style="color:#888;">|</font>&nbsp;

      &nbsp;&nbsp;<span style="font-family: sans-serif;font-size: 12px;text-transform: uppercase;">Explore:</span>&nbsp;&nbsp;
      <a href="list/" class="menu">&nbsp;Lists&nbsp;</a>
      <!--<a href="sim/" class="menu">&nbsp;Similar Forms&nbsp;</a>
      <a href="flow/" class="menu">&nbsp;Flows&nbsp;</a>-->
      
        </p>
    </div>

  <div class="content"><h2><a name="title" href="#title" class="anchor"></a>What is the Form Explorer?</h2>

    <div style="float:left;width:100%%">
      <div class="r_img_embed" style="max-width:500px;">
        <img src="images//bigdata.png" alt="Image of the character Data as a giant standing next to the Supreme Court" width="100%%"/>
        <div class="caption">Big Data &amp; the Law, h/t <a href="https://flic.kr/p/7YZubW" target="_blank">Tim Sackton</a> &amp; <a href="https://medium.com/@wtrsld/big-data-made-me-do-it-5bfc3f46871c" target="_blank">Josh Lee</a></div>
      </div>
      <p>
        We're collecting existing pdf forms from multiple jurisdictions and making it possible for folks (e.g., courts and non-profits) to turn them into beautiful webapps with ease. Eventually, you will be able to go from form to prototype webapp in a few clicks. As time goes by, we'll grow the number of jurisdictions and improve our tools' performance. They'll never do all of the work, but they help plot a course for what otherwise might seem an "impossible" task. They'll help make things concrete, allowing for stakeholders and developers to edit their way to a production-ready solution built on suggestions rooted in years of form building experience.
      </p>
      <p>
        For many, interaction with a court involves completing a "form," and often, this means filling out either a paper form or pdf facsimile. Occasionally, the latter can be found online and come with embedded fillable fields. Most of the time, however, this just means one can type in the fields before printing the form. The infrastructure that has grown up around court forms too often focus on delivering either a physical form or pictures of such to the court. Very rare is the form that can easily be filled out on a phone absent specialty software, and even more rare is the form that helps guide users through the process. 
        There are exceptions, like the forms found at <a href="https://lawhelpinteractive.org/" target="_blank">LawHelp Interactive</a> or the growing constellation of <a href="https://www.a2jauthor.org/" target="_blank">A2J Author</a>-driven forms found across multiple jurisdictions. Yet, the majority of court processes linked to forms have failed to keep pace with our technical ability to offer context-aware mobile friendly fully-electronic interactions. This Form Explorer is part of a broader open-source effort—<a href="https://suffolklitlab.org/docassemble-AssemblyLine-documentation/" target="_blank">the Document Assembly Line</a>—aimed at changing this. To learn more about how we got started, check out <a href="https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3911381" target="_blank">Digital Curb Cuts: Towards an Inclusive Open Forms Ecosystem</a>, a law review article describing the project's genesis.
      </p>
      <p>
        On this site you can explore the connections between forms and get a feel for populations of existing forms, allowing you to strategize how best to redesign or translate flat forms into vibrant interactive web apps. 
      </p>
      <h2><a name="data" href="#data" class="anchor"></a>Our Data</h2>
      <p>
        Currently, we've downloaded and parsed %s forms from %s jurisdictions. Use the menu above to explore, or download summary data here: <a href="https://courtformsonline.org/forms/form_data.csv" target="_blank">form_data.csv</a> And please remember this is a work in progress. So, everything here is subject to change including things like fields we find and how we calculate readability. 
      </p>
      <h2><a name="data" href="#tools" class="anchor"></a>Our Tools</h2>
      <p>
        To mine data from these forms we created an open source python package—FormFyxer. You can find it and its documentaion on GitHub here: <a href="https://github.com/SuffolkLITLab/FormFyxer" target="_blank">https://github.com/SuffolkLITLab/FormFyxer</a>
      </p>
    </div>"""%(len(tmp_df),len(tmp_df["jurisdiction"].unique()))   
    return html

In [344]:
html = header("About the Form Explorer?","""We're collecting existing pdf forms from multiple jurisdictions and making it possible for folks (e.g., courts and non-profits) to turn them into beautiful webapps with ease. Eventually, you will be able to go from form to prototype webapp in a few clicks. As time goes by, we'll grow the number of jurisdictions and improve our tools' performance. They'll never do all of the work, but they help plot a course for what otherwise might seem an "impossible" task. They'll help make things concrete, allowing for stakeholders and developers to edit their way to a production-ready solution built on suggestions rooted in years of form building experience.""")+home()+footer()
text_file = open("docs/index.html", "w")
text_file.write(html)
text_file.close()

# redirects

In [345]:
def redirects(jur):
    html = """<html>
<head>
<META http-equiv="CACHE-CONTROL" CONTENT="NO-CACHE">
<meta http-equiv="refresh" content="0; url=%s" />
</head>
</html>
"""%jur
    return html

In [346]:
html = redirects(default_jur)

#text_file = open("docs/sim/index.html", "w")
#text_file.write(html)
#text_file.close()

#text_file = open("docs/flow/index.html", "w")
#text_file.write(html)
#text_file.close()

#text_file = open("docs/list/index.html", "w")
#text_file.write(html)
#text_file.close()

# List VIew

In [369]:
def listview(jur,path="./"):
    
    pages = files_df[(files_df["jurisdiction"]==jur[0])]["pages"].mean()
    f_per_p = files_df[(files_df["jurisdiction"]==jur[0])]["f_per_p"].mean()
    fields = files_df[(files_df["jurisdiction"]==jur[0])]["f_per_p"].mean()*files_df[(files_df["jurisdiction"]==jur[0])]["pages"].mean()
    reading = files_df[(files_df["jurisdiction"]==jur[0])]["reading"].median()
    forms= len(files_df[(files_df["jurisdiction"]==jur[0])])
    
    html = """
    <div class="menu_bar">
        <p style="text-align:center;">
      <a href="https://suffolklitlab.org/" class="menu">&nbsp;LIT Lab Home&nbsp;</a>&nbsp;<font style="color:#888;">|</font>&nbsp;
      <a href="../../" class="menu">&nbsp;About&nbsp;The&nbsp;Explorer&nbsp;</a>&nbsp;<font style="color:#888;">|</font>&nbsp;
      <a href="../../compare" class="menu">Find&nbsp;&amp;&nbsp;Compare&nbsp;</a><font style="color:#888;">|</font>&nbsp;

      &nbsp;&nbsp;<span style="font-family: sans-serif;font-size: 12px;text-transform: uppercase;">Explore:</span>&nbsp;&nbsp; <select style="width:100px;" onChange="window.location.href='../'+this.value">"""
    
    for item in jurs: 
        if item[0]==jur[0]:
            selected = "SELECTED"
        else:
            selected = ""
        html += "<option value=\"%s\" %s>%s</option>"%(item[0],selected,item[1])
        
    html += """
      </select>&nbsp;&nbsp;
      <a href="../../list/%s" class="menu" style="color:black;">&nbsp;Lists&nbsp;</a>
      <!--<a href="../../sim/%s" class="menu">&nbsp;Similar Forms&nbsp;</a>
      <a href="../../flow/%s" class="menu">&nbsp;Flows&nbsp;</a>-->
        </p>
    </div>
"""%(jur[0],jur[0],jur[0])
    
    html += """
    <div class="content">
    <h2><a name="title" href="#title" class="anchor"></a>List of %s Court Forms</h2>
    
    <div class="sidebar">
        <h2>Search %s Forms</h2>
        <table cellpadding="0px"><tr><td width="100%%">
        <input id="q" style="width:100%%">
        </td><td width="1%%">&nbsp;&nbsp;&nbsp;</td><td width="1%%">
        <input type="button" value="Search" onClick="open_search();">
        </td></tr></table>
        <script>
            sort_links();

            function open_search(){
                alert('Coming Soon!');
            }
            function open_search_(){
                window.open('https://www.google.com/search?q='+$('#q').val()+'+site:suffolklitlab.org/form-explorer/form/%s')
            }
            $("#q").on('keyup', function (e) {
                if (e.key === 'Enter' || e.keyCode === 13) {
                    open_search()
                }
            });
        </script>
    </div>

    <p>
    Below you will find %s unique forms. The average form is %.0f pages long, with %.0f fields per page. The median form is written at a %.2f-grade reading level.
    </p>
    
    <p>Download a .csv file with meta data on all %s forms, including source urls, field names, reading levels, et al.: 
    <a href="../../data/%s_form_data.csv" target="_blank">%s_form_data.csv</a>
    </p>
    

    <div id="court_grouping" style="display:none;width:100;">
    
    <p><i>Links are grouped by categories found on the court's website.</i></p>
    
    """%(jur[1],jur[1],jur[0],forms,pages,fields,reading,forms,jur[0],jur[0])

    #for cat in files_df[files_df["jurisdiction"]==jur[0]]["group"].unique():
    #    html += """
    #    <h3><a name="%s" href="#%s" class="anchor"></a>%s</h3>
    #    <ul>"""%(removeSpecial(cat),removeSpecial(cat),cat)

    #    for index,row in files_df[(files_df["jurisdiction"]==jur[0]) & (files_df["group"]==cat)].iterrows():
            #print(row[1][0])
    #        html += """<li><a href="../../form/%s/%s.html">%s</a></li>"""%(jur[0],row["id"],row["title"])
            
    #    html += "</ul><p style=\"text-align:center\"><a href=\"#title\">go to top</a></p>"
        
    html+="""</div>
    
    <div id="LIST_grouping" style="width:100;">
    
    <p><i>Links are grouped by categories from the <a href="https://taxonomy.legal/" target="_blank" class="exlink">Legal Issue Taxonomy (LIST)</a>. 
    Note: the following grouping was done by machine. 
    """

    list_group = files_df[(files_df["list"]=="[]") & (files_df["jurisdiction"]==jur[0])]
    if len(list_group)>0:
        html+="""
        Forms without an identifed group can befound under the <a href="#unclassifed">Unclassified</a> header.</i></p>
        """
    
    lookfor = [
                ["BE","Public Benefits"],
                ["WO","Work and Employment Law"],
                ["ES","Estates, Wills, and Gaurdianships"],
                ["CR","Crime and Prisons"],
                ["FA","Family"],
                ["TO","Accidents and Torts"],
                ["TR","Trafic and Cars"],
                ["HE","Health"],
                ["MO","Money, Debt, and Consumer Issues"],
                ["HO","Housing"],
                ["VE","Veterans and Military"],
                ["IM","Immigration"],
                ["DI","Disaster Relief"],
                ["RI","Civil and Human Rights"],
                ["EN","Environmental Justice"],
                ["NA","Native American Issues and Tribal Law"],
                ["ED","Education"],
                ["BU","Small Business and IP"],
                ["GO","Government Services"],
                ["CO","Courts and Lawyers"],
              ]
    
    for item in lookfor:
        list_group = files_df[(files_df["list"].str.contains(item[0])) & (files_df["jurisdiction"]==jur[0]) & (files_df["list"]!="[]")]
        if len(list_group)>0:
            html+="<h3><a name=\"%s\" href=\"#%s\" class=\"anchor\"></a>%s</h3><ul>"%(item[1],item[1],item[1])
            for index,row in list_group.iterrows():
                html += """<li><a href="../../form/%s/%s.html">%s</a></li>"""%(jur[0],row["id"],row["title"])
            html+="</ul>"
            html+="""<p style="text-align:center"><a href="#title">go to top</a></p>"""
        #else:
        #    html+="<h3 style=\"color:gray\">%s</h3>"%item[1]
        
        
    list_group = files_df[(files_df["list"]=="[]") & (files_df["jurisdiction"]==jur[0])]
    if len(list_group)>0:
        html+="<h3><a name=\"unclassifed\" href=\"#unclassifed\" class=\"anchor\"></a>Unclassified</h3><ul>"
        for index,row in list_group.iterrows():
            html += """<li><a href="../../form/%s/%s.html">%s</a></li>"""%(jur[0],row["id"],row["title"])
        html+="</ul>"
        html+="""<p style="text-align:center"><a href="#title">go to top</a></p>"""
        
    html+="""
    </div>
    
    <div id="most_used" style="display:none;width:100;">
    
    <p><i>Links are orderd by most used.</i></p>
    
    <p style="background:#fcf19d">
        <i>
            <b>Under Construction:</b> With historical form data we could provde insights on how frequently forms are used.
            If you work with the court and can help provide historic form data, <a href="mailto:litlab@suffolk.edu">let us know</a>.
        </i>
    </p>
    
    </div>
    """
        
    #for field in included_fields:
    #    print("\t"+field)
    #    html += """<li><a href="fields/%s.html">%s</a></li>"""%(field,field)
        
    html+="</div>"

    return html

In [370]:
files_df['f_per_p'] = files_df['f_per_p'].fillna(0)

In [371]:
#files_df[files_df['f_per_p'].str.contains("brevity")]

In [372]:
files_df[files_df['f_per_p']=="['throughout_word_used_brevity', 'page_field__1', 'page_field__2', 'page_field__3', 'page_field__4', 'page_field__5', 'page_field__6', 'page_field__7', 'page_field__8', 'page_field__9', 'page_field__10', 'page_field__11', 'page_field__12', 'page_field__13', 'page_field__14', 'page_field__15', 'page_field__16', 'page_field__17', 'page_field__18', 'page_field__19', 'page_field__20', 'page_field__21', 'page_field__22', 'page_field__23', 'page_field__24', 'page_field__25', 'page_field__26', 'page_field__27', 'page_field__28', 'page_field__29', 'montana_university_bozeman_mt']"]

,id,jurisdiction,source,title,group,url,filename,downloaded,pages,fields,fields_conf,fields_old,f_per_p,reading,list,text,downloaded_on


In [373]:
test = files_df.copy()
test = test.drop([0, 9087])
#test.loc[9087]

KeyError: '[   0 9087] not found in axis'

In [374]:
files_df = files_df.drop([0, 9087])

KeyError: '[   0 9087] not found in axis'

In [354]:
#files_df["f_per_p"].mean()

In [375]:
files_df["f_per_p"] = pd.to_numeric(files_df["f_per_p"])#.mean()

In [376]:
for jur in jurs:    
    print(jur)
    path = "../../"
    
    if not os.path.exists("docs/list/%s"%(jur[0])):
        os.makedirs("docs/list/%s"%(jur[0]))
    #else:
    #    shutil.rmtree("docs/list/%s"%(jur[0]))
    #    os.makedirs("docs/list/%s"%(jur[0]))
        
    html = header("%s Court Forms"%jur[1],"""We're collecting existing pdf forms from multiple jurisdictions and making it possible for folks (e.g., courts and non-profits) to turn them into beautiful webapps with ease. Eventually, you will be able to go from form to prototype webapp in a few clicks. As time goes by, we'll grow the number of jurisdictions and improve our tools' performance. They'll never do all of the work, but they help plot a course for what otherwise might seem an "impossible" task. They'll help make things concrete, allowing for stakeholders and developers to edit their way to a production-ready solution built on suggestions rooted in years of form building experience.""",path,wide=0)+listview(jur,path)+footer(path)
    text_file = open("docs/list/%s/index.html"%(jur[0]), "w", encoding="utf-8")
    text_file.write(html)
    text_file.close()

['AL', 'Alabama']
['AK', 'Alaska']
['AR', 'Arkansas']
['CA', 'California']
['CO', 'Colorado']
['CT', 'Connecticut']
['DC', 'District of Columbia']
['DE', 'Delaware']
['FL', 'Florida']
['GA', 'Georgia']
['HI', 'Hawaii']
['ID', 'Idaho']
['IL', 'Illinois']
['IA', 'Iowa']
['KS', 'Kansas']
['KY', 'Kentucky']
['LA', 'Louisiana']
['ME', 'Maine']
['MD', 'Maryland']
['MA', 'Massachusetts']
['MI', 'Michigan']
['MN', 'Minnesota']
['MS', 'Mississippi']
['MT', 'Montana']
['NE', 'Nebraska']
['NV', 'Nevada']
['NJ', 'New Jersey']
['NY', 'New York']
['NC', 'North Carolina']
['ND', 'North Dakota']
['OH', 'Ohio']
['OK', 'Oklahoma']
['OR', 'Oregon']
['PA', 'Pennsylvania']
['RI', 'Rhode Island']
['SC', 'South Carolina']
['SD', 'South Dakota']
['TN', 'Tennessee']
['TX', 'Texas']
['UT', 'Utah']
['VT', 'Vermont']
['VA', 'Virginia']
['WA', 'Washington']
['WV', 'West Virginia']
['WI', 'Wisconsin']
['WY', 'Wyoming']
['ca1', 'First Circuit']


# Create form pages

In [377]:
def formview(jur,path="./"):
    html = """
    <div class="menu_bar">
        <p style="text-align:center;">
      <a href="https://suffolklitlab.org/" class="menu">&nbsp;LIT Lab Home&nbsp;</a>&nbsp;<font style="color:#888;">|</font>&nbsp;
      <a href="../../" class="menu">&nbsp;About&nbsp;The&nbsp;Explorer&nbsp;</a>&nbsp;<font style="color:#888;">|</font>&nbsp;
      <a href="../../compare" class="menu">Find&nbsp;&amp;&nbsp;Compare&nbsp;</a><font style="color:#888;">|</font>&nbsp;
      &nbsp;&nbsp;<span style="font-family: sans-serif;font-size: 12px;text-transform: uppercase;">Explore: """
    
    for item in jurs: 
        if item[0]==jur[0]:
            this_state = item[1]
            html += "%s"%this_state

    html += """
      </span>&nbsp;&nbsp;
      <a href="../../list/%s" class="menu">&nbsp;Lists&nbsp;</a>
      <!--<a href="../../sim/%s" class="menu">&nbsp;Similar Forms&nbsp;</a>
      <a href="../../flow/%s" class="menu">&nbsp;Flows&nbsp;</a>-->
        </p>
    </div>

"""%(jur[0],jur[0],jur[0])
    
    if jur[0]!="CA":
        linkurl = "https://courtformsonline.org/forms/%s.pdf"%row["id"]
        iframelink = linkurl
    else:
        linkurl = "javascript:alert('Califonia machine-processed forms are not yet available.')"
        iframelink = "https://courtformsonline.org/forms/%s.pdf"%row["id"]
        
    html += """
  <div class="content_wide" style="padding-left:0px;">
    <div class="pdf">
      <iframe src="%s" width="100%%" height="1100px;"></iframe>
      <div id="form_text" style="display:none;overflow:hidden;padding:20px;">Here is the text we could read: 
      <pre>%s</pre></div>
    </div>
    <div class="pdfinfo">
      <h1><a name="title" href="#title" class="anchor"></a>%s</h1>
      <p>
        This info page is part of the LIT Lab's <i>Form Explorer</i> project. It is not associated with the %s state courts. 
        To learn more about the project, check out our <a href="../../">about page</a>.
      </p>
      <p>
        <b>Downloads:</b> You can download both the <a href="%s" target="_blank" class="exlink">original form</a> (last checked %s) 
        and the <a href="%s">machine-processed form</a> with normalized data fields.
      </p>

        <hr style="border: 1px solid #fff;border-bottom: 1px solid #555;">
      <h3><a name="about_form" href="#about_form" class="anchor"></a>About This Form:</h3>
      <ul>"""%(iframelink,row["text"],row["title"],this_state,row["url"],row["downloaded"],linkurl)
    
    html += """
        <li>Sourced from <a href="http://%s" target="_blank" class="exlink">%s</a> (%s)</li>
        <li>Page(s): %s </li>
        <li>Fields(s): %s </li>
        <li>Average fields per page: %s</li>
        <li>Reading Level: Grade %s</li>
        <li><a href="https://taxonomy.legal/" target="_blank" class="exlink">LIST</a> Grouping(s): 
    """%(row["source"],row["source"],row["downloaded"],clean_n(row["pages"]),clean_n(row["pages"]*row["f_per_p"]),clean_n(row["f_per_p"]),clean_n(row["reading"]))
    
    if pd.notnull(row["list"]):
        j= 0
        nsmi = eval(row["list"])
        if len(nsmi)>0:
            for f in nsmi:
                #html += "<a href=\"https://taxonomy.legal/term/%s\" target=\"_blank\">%s</a>"%(f,f)
                html += "%s"%(f)
                if j==len(nsmi)-1:
                    html+=". "
                else:
                    html+=", "
                j+=1    
        else:
            html+="Unknown"
    else:
            html+="Unknown"
            nsmi = []
        
    html += """        
        </li>
      </ul>
      <p>
      <i>Use our <a href="https://ratemypdf.com" target="_blank">Rate My PDF</a> tool to learn more.</i> 
      Go beyond the above insights and learn more about this or any pdf form at 
      <a href="https://ratemypdf.com" target="_blank">RateMyPDF.com</a>, includes: counts of difficult words used, 
      passive voice decetion, and suggestions for how to make the form more usable.
      </p>
      <hr style="border: 1px solid #fff;border-bottom: 1px solid #555;">
      <h3><a name="fields" href="#fields" class="anchor"></a>Identified Data Fields:</h3>
      <p>We have done our best to automaticly identify and name form fields according to our <a href="https://suffolklitlab.org/docassemble-AssemblyLine-documentation/docs/label_variables/">naming conventions</a>. 
      When possible, we've used names tied to our question library. See e.g., <a href="https://suffolklitlab.org/docassemble-AssemblyLine-documentation/docs/question_library/names">user1_name</a>.
      If we think we've found a match to a question in our library, it is highlighted in green. Novel names are auto generated. So, you will probably need to edit some of them if you're trying to stick to the convention.
      </p>
      
      <div style="float:left;width:100%%;margin-bottom:25px;">
          <div class="tab" style="border: 1px solid #fff;border-bottom: 1px solid #555;width:15px;padding:10px 0px;">&nbsp;</div>
          <div class="tab" id="tab1" style="border-bottom: 1px solid #fff;" >
              <a href="javascript:void('');" onClick="tab_focus('orig_fields');" class="menu" id="atab1" style="color:black;">Original Order</a>
          </div>
          <div class="tab" style="border: 1px solid #fff;border-bottom: 1px solid #555;width:15px;padding:10px 0px;">&nbsp;</div>
          <div class="tab" id="tab2">
              <a href="javascript:void('');" onClick="tab_focus('sug_screens');" class="menu" id="atab2">Suggested Screens</a>
          </div>
          <div class="tab" style="border: 1px solid #fff;border-bottom: 1px solid #555;">
              &nbsp;
          </div>
      </div>
      <script>
          function tab_focus(tab){
              if (tab=="orig_fields") {
                  $('#sug_screens').hide();
                  $('#orig_fields').show();
                  $('#tab1').css('border-bottom-color', '#fff;');
                  $('#tab2').css('border-bottom-color', '#555;');
                  $('#atab1').css('color', 'black');
                  $('#atab2').css('color', '#529ecc');
              } else {
                  $('#orig_fields').hide();
                  $('#sug_screens').show();
                  $('#tab1').css('border-bottom-color', '#555;');
                  $('#tab2').css('border-bottom-color', '#fff;');
                  $('#atab1').css('color', '#529ecc');
                  $('#atab2').css('color', 'black');
              }
          }
      </script>
      
      """%()
    
    
    html+="<div id=\"orig_fields\" style=\"padding-left:15px;\"><p>Here are the fields we could identify.</p>"
    html+="<div style=\"max-height:550px;overflow-y: auto;padding-bottom:25px;\"><ul>"
        
    if not pd.isnull(row["fields"]):
        if len(eval(row["fields"]))==0:
            html+="<li>No fields found</li>"
        else:        
            j = 0
            for f in eval(row["fields"]):
                html += "<li>"
                if re.search('^\*',f):
                    f = re.sub("^\*","",f)
                    f_link = re.sub("__\d+$","",f)
                    #v = re.search('^(\w*)',f).groups()[0]
                    #f = re.sub("\|","</code></a>",f)
                    html +="<code style=\"background:#a2e874\"><a href=\"../../list/%s/fields/%s.html\">%s</a></code> was <i>%s</i> (%.2f conf)<!--100%% completed. <a href=\"\">Learn more about distribution of answers.</a>--></li>"%(jur[0],f_link,f,eval(row["fields_old"])[j],eval(row["fields_conf"])[j])   
                else: 
                    #v = re.search('^(\w*)',f).groups()[0]
                    #f = re.sub("\|","</code>",f)
                    html +="<code>%s</code> was <i>%s</i> (%.2f conf)<!--100%% completed. <a href=\"\">Learn more about distribution of answers.</a>--></li>"%(f,eval(row["fields_old"])[j],eval(row["fields_conf"])[j])
                j+=1    
                
    else:
        html+="<li>No fields found</li>"

    html+="</ul></div></div>"
        
    html+="<div id=\"sug_screens\" style=\"display:none;padding-left:15px;\"><p>We've done our best to group similar variables togther to avoid overwhelming the user.</p>"
    html+="<div style=\"max-height:550px;overflow-y: auto;padding-bottom:25px;\">"

    if not pd.isnull(row["fields"]):
        if len(eval(row["fields"]))==0:
            html+="<ul><li>No fields found</li>"
        else:       
            screens = lit.cluster_screens(eval(row["fields"]),damping=0.5)
            i = 0
            for screen in screens:
                html+="<p>Suggested Screen %s:</p><ul>"%i
                j = 0
                for f in screens[screen]:
                    html += "<li>"
                    if re.search('^\*',f):
                        f = re.sub("^\*","",f)
                        f_link = re.sub("__\d+$","",f)
                        #v = re.search('^(\w*)',f).groups()[0]
                        #f = re.sub("\|","</code></a>",f)
                        html +="<code style=\"background:#a2e874\"><a href=\"../../list/%s/fields/%s.html\">%s</a></code></li>"%(jur[0],f_link,f)   
                    else: 
                        #v = re.search('^(\w*)',f).groups()[0]
                        #f = re.sub("\|","</code>",f)
                        html +="<code>%s</code></li>"%(f)
                    j+=1    
                i+=1
                html+="</ul>"
    else:
        html+="<li>No fields found</li>"    

    html+="</div></div>"

        
    html += """
      
      <hr style="border: 1px solid #fff;border-bottom: 1px solid #555;">
      <h3><a name="weaver" href="#weaver" class="anchor"></a>Create an Interactive Version of this Form:</h3>
      <p>
      The Weaver creates a draft guided interview from a template form, like the one provided here.
      To learn more, read <a href="https://suffolklitlab.org/docassemble-AssemblyLine-documentation/docs/generating_code/">"Weaving" your form into a draft interview</a>.
      </p>"""
    
    html+="""
    </div>
  </div>
    """
    return html

In [85]:
# run one sample
path = "../../"
for index,row in files_df[files_df["id"]=="6b4ebd487f82f387512ac20da28803db"].iterrows():
    print(row["jurisdiction"],row["title"])
    
    html = header(row["jurisdiction"]+": "+row["title"],"""We're collecting existing pdf forms from multiple jurisdictions and making it possible for folks (e.g., courts and non-profits) to turn them into beautiful webapps with ease. Eventually, you will be able to go from form to prototype webapp in a few clicks. As time goes by, we'll grow the number of jurisdictions and improve our tools' performance. They'll never do all of the work, but they help plot a course for what otherwise might seem an "impossible" task. They'll help make things concrete, allowing for stakeholders and developers to edit their way to a production-ready solution built on suggestions rooted in years of form building experience.""",path,wide=1)+formview([row["jurisdiction"],""],path)+footer(path)
    text_file = open("docs/form/%s/%s.html"%(row["jurisdiction"],row["id"]), "w", encoding="utf-8")
    
    text_file.write(html)
    text_file.close()

AL Workers' Compensation Notice


/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/sklearn/cluster/_affinity_propagation.py:144: UserWarning: All samples have mutually equal similarities. Returning arbitrary cluster center(s).
  warnings.warn(


In [380]:
%%time

path = "../../"

# comment out if only backfilling
if 1==2:
    for jur in jurs:    
        if not os.path.exists("docs/form/%s"%(jur[0])):
            os.makedirs("docs/form/%s"%(jur[0]))
        else:
            shutil.rmtree("docs/form/%s"%(jur[0]))
            os.makedirs("docs/form/%s"%(jur[0]))
        
print("Start")

#loop_df = files_df[files_df["jurisdiction"].isin(list(np.array(jurs)[:, 0]))]
loop_df = files_df[files_df["jurisdiction"]=="ca1"] #backfill
i = 0
p_1 = round(len(loop_df)/100)
for index,row in loop_df.iterrows():
    
    if not os.path.exists("docs/form/%s/%s.html"%(row["jurisdiction"],row["id"])):
        print("{}) {}: {}".format(i,row["jurisdiction"],row["title"]))
        html = header(row["jurisdiction"]+": "+row["title"],"description goes here",path,wide=1)+formview([row["jurisdiction"],""],path)+footer(path)
        text_file = open("docs/form/%s/%s.html"%(row["jurisdiction"],row["id"]), "w", encoding="utf-8")
        text_file.write(html)
        text_file.close()
        
    i += 1
    if i%p_1==0:
        print("\n{}% done...".format(i/p_1))
        governor(0,50,100,1)
    else:
        governor(0,50,100,0)    

print("END")

Start
0) ca1: Bill of Costs


/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/sklearn/cluster/_affinity_propagation.py:250: ConvergenceWarning: Affinity propagation did not converge, this model will not have any cluster centers.
  warnings.warn(
/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/sklearn/cluster/_affinity_propagation.py:528: ConvergenceWarning: This model does not have any cluster centers because affinity propagation did not converge. Labeling every sample as '-1'.
  warnings.warn(



1.0% done...

 Usage: CPU 11.5% & Memory 66.4%

1) ca1: U.S. Marshals Service Form 285 - Service of Process

2.0% done...

 Usage: CPU 52.9% & Memory 66.3%
    Slowing down...

2) ca1: U.S. Marshals Service Form 285 - Service of Process

3.0% done...

 Usage: CPU 57.5% & Memory 66.3%
    Slowing down...

3) ca1: Step by Step Guide to Filing a Civil Action Pro Se _ United States District Court for the District of Massachusetts

4.0% done...

 Usage: CPU 0.0% & Memory 66.3%

4) ca1: SS_Complaint

5.0% done...

 Usage: CPU 0.0% & Memory 66.3%

5) ca1: Return Bail Bond Documents (Spanish)

6.0% done...

 Usage: CPU 0.0% & Memory 66.3%

6) ca1: Response_to_Motion

7.0% done...

 Usage: CPU 0.0% & Memory 66.3%

7) ca1: RequestforMediation

8.0% done...

 Usage: CPU 0.0% & Memory 66.3%

8) ca1: Request for Hearing re discovery 26b

9.0% done...

 Usage: CPU 61.3% & Memory 66.4%
    Slowing down...

9) ca1: Reply_to_Response_to_Motion

10.0% done...

 Usage: CPU 64.3% & Memory 66.4%
    Slowi

/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/sklearn/cluster/_affinity_propagation.py:250: ConvergenceWarning: Affinity propagation did not converge, this model will not have any cluster centers.
  warnings.warn(
/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/sklearn/cluster/_affinity_propagation.py:528: ConvergenceWarning: This model does not have any cluster centers because affinity propagation did not converge. Labeling every sample as '-1'.
  warnings.warn(



12.0% done...

 Usage: CPU 0.0% & Memory 66.4%

12) ca1: Complaint to Require Performance of a Contract to Convey Real Property


...

/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/sklearn/cluster/_affinity_propagation.py:250: ConvergenceWarning: Affinity propagation did not converge, this model will not have any cluster centers.
  warnings.warn(
/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/sklearn/cluster/_affinity_propagation.py:528: ConvergenceWarning: This model does not have any cluster centers because affinity propagation did not converge. Labeling every sample as '-1'.
  warnings.warn(



93.0% done...

 Usage: CPU 62.9% & Memory 66.4%
    Slowing down...

93) ca1: Petition for Relief from a Conviction or Sentence


/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/sklearn/cluster/_affinity_propagation.py:250: ConvergenceWarning: Affinity propagation did not converge, this model will not have any cluster centers.
  warnings.warn(
/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/sklearn/cluster/_affinity_propagation.py:528: ConvergenceWarning: This model does not have any cluster centers because affinity propagation did not converge. Labeling every sample as '-1'.
  warnings.warn(



94.0% done...

 Usage: CPU 25.9% & Memory 66.5%

94) ca1: Application to Proceed in District Court Without Prepaying Fees or Costs (Short Form)

95.0% done...

 Usage: CPU 20.4% & Memory 66.5%

95) ca1: Application to proceed in District Courts Without Prepaying Fees or Costs (long form)

96.0% done...

 Usage: CPU 32.0% & Memory 66.5%

96) ca1: 2254 Instructions to Inmates (English Spanish)

97.0% done...

 Usage: CPU 56.5% & Memory 66.5%
    Slowing down...

97) ca1: 1983 Notice to Inmates (English Spanish)

98.0% done...

 Usage: CPU 79.5% & Memory 66.6%
    Slowing down...

98) ca1: 1983 Instructions to Inmates (English Spanish)

99.0% done...

 Usage: CPU 0.0% & Memory 66.5%

99) ca1: 1331 Instructions to Inmates (English)

100.0% done...

 Usage: CPU 0.0% & Memory 66.5%

100) ca1: untitled

101.0% done...

 Usage: CPU 0.0% & Memory 66.5%

END
CPU times: user 1min 35s, sys: 1.32 s, total: 1min 37s
Wall time: 31 s


/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/sklearn/cluster/_affinity_propagation.py:250: ConvergenceWarning: Affinity propagation did not converge, this model will not have any cluster centers.
  warnings.warn(
/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/sklearn/cluster/_affinity_propagation.py:528: ConvergenceWarning: This model does not have any cluster centers because affinity propagation did not converge. Labeling every sample as '-1'.
  warnings.warn(
/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/sklearn/cluster/_affinity_propagation.py:144: UserWarning: All samples have mutually equal similarities. Returning arbitrary cluster center(s).
  warnings.warn(
/Users/davidcolarusso/opt/anaconda3/lib/python3.8/site-packages/sklearn/cluster/_affinity_propagation.py:144: UserWarning: All samples have mutually equal similarities. Returning arbitrary cluster center(s).
  warnings.warn(
/Users/davidcolarusso/opt/anaconda3/lib/python3.

In [283]:
list(np.array(jurs)[:, 0])

['MA']

In [381]:
hashme("asd")

'7815696ecbf1c96e6894b779456d330e'

In [384]:
# importing os module
import os
 
# Function to rename multiple files
def main():
   
    folder = "../data/fed_forms/all_together_now"
    for count, filename in enumerate(os.listdir(folder)):
        #dst = f"Hostel {str(count)}.jpg"
        src =f"{folder}/{filename}"  # foldername/filename, if .py file is outside folder
        dst =f"{folder}/{hashme(filename)}.pdf"
         
        # rename() function will
        # rename all the files
        os.rename(src, dst)

In [385]:
main()